# SignSight — ZIP 107 lớp → CAR35 → train YOLO11s

Dùng ZIP SupSign/YOLO **107 lớp**, có data.yaml và train/valid/test ở thư mục gốc.
Chỉ cần một link gdown; không mount Drive, không clone repo, không cần upload dataset mới.
Notebook tự kiểm tra ảnh/nhãn, lọc và remap 35 lớp CAR35, loại bản trùng byte,
gom ảnh gần trùng bằng dHash rồi chia nhóm theo mục tiêu 70/15/15.
Ảnh chỉ chứa lớp ngoài CAR35 được giữ làm nền đối với subset.

Đây là **dataset/split mới**, không phải V4/R3 cũ và không tự áp patch R3 khóa theo manifest cũ.
Không so trực tiếp mAP giữa hai split. dHash không chứng minh độc lập chuyến xe/thực địa.
Xem EDA và nhãn mẫu trước khi chạy cell train. Notebook không đổi model trong app.


In [ ]:
import sys, subprocess, os
os.environ['YOLO_AUTOINSTALL'] = 'false'
os.environ['WANDB_MODE'] = 'disabled'
packages = ['Pillow>=10,<13', 'matplotlib>=3.8,<4', 'PyYAML>=6,<7', 'gdown==5.2.0', 'tqdm>=4.66,<5']
packages.append("ultralytics==8.4.173")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])


## 1. Cấu hình

Dán link file ZIP 107 lớp vào SOURCE_URL. Không điền checksum của ZIP V4 vào đây.
RAW_ZIP_SHA256 có thể để trống khi chưa có hash đáng tin cậy; hash thực tế luôn được ghi nhận.
LOCAL_SOURCE_ZIP dùng nếu ZIP raw vẫn còn trong runtime: điền đường dẫn file để khỏi tải lại.
Nếu helper cũ đã xóa file khi báo lỗi SHA, cần tải lại bằng gdown một lần.

Mọi file nằm trong /content; không tự lưu lên Drive. Cần đủ ổ cho ZIP, raw giải nén và CAR35.
Thư mục raw tạm do cell tạo sẽ được dọn sau khi chuẩn bị; ZIP gốc được giữ nguyên.
Đổi DATASET_VERSION khi đổi dữ liệu/seed/grouping/code; cache cũ không bị ghi đè.


In [ ]:
from pathlib import Path
import sys, os
WORK = Path('/content/signsight_raw107')
WORK.mkdir(parents=True, exist_ok=True)
(WORK / '.ultralytics').mkdir(exist_ok=True)
os.environ['YOLO_CONFIG_DIR'] = str(WORK / '.ultralytics')
SOURCE_URL = ''  # Link file ZIP 107 lớp trên Drive
LOCAL_SOURCE_ZIP = ''  # Tùy chọn: đường dẫn ZIP raw đã có trong runtime
RAW_ZIP_SHA256 = ''  # Tùy chọn; không dùng SHA của CAR35 V4
DATASET_VERSION = 'car35_raw107_v1'
DATASET_ROOT = WORK / DATASET_VERSION
SEED = 42
NEAR_HASH_DISTANCE = 4
WORKERS = 4
RUN_NAME = 'yolo11s_' + DATASET_VERSION + '_640'
RUN_DIR = WORK / 'runs' / RUN_NAME
EPOCHS = 30
IMAGE_SIZE = 640
BATCH = 8
DEVICE = '0'
RESUME = False
RUN_HELDOUT_TEST = False
DOWNLOAD_RESULTS = True
assert DATASET_VERSION and Path(DATASET_VERSION).name == DATASET_VERSION and DATASET_VERSION not in ('.','..')
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in ('.','..')
print('Dataset:', DATASET_ROOT)


In [ ]:
# Helpers đã nhúng; chạy lại cell này để thay module cũ trong cùng runtime.
import hashlib, importlib, json, shutil
TRAINING_FILES = {'training/__init__.py': '"""Offline datasets, experiments and notebook tooling; never imported by the API."""\n', 'training/compatibility.py': '"""Explicit cache provenance checks; content identity is separate from file checksum."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nLEGACY_SIGNATURES = {\n (\'42132e8eb337a7d8c27ef35eb5b81688f879d01afaccf593fa88265c435f4eba\',\n  \'f89bb8e458fc951cb997f2de1e57497eeaf61f97a5824e179b66b33bde3e1e03\'),\n (\'ca8ef48fb483c142085aae0dc6ff9bb570074eac968c686ad4ec2ae4672bbf51\',\n  \'1893e87f0b5fff1719b1590a1bbbb511c1dd40714cf8e3ff4c7bf760eb5b91b1\'),\n}\n\ndef implementation_digest():\n    root=Path(__file__).parent\n    names=[\'compatibility.py\',\'data/common.py\',\'data/supsign.py\',\'data/cache.py\',\n           \'experiments/train.py\',\'experiments/evaluate.py\']\n    content={name:(root/name).read_text(encoding=\'utf-8\') for name in names}\n    return hashlib.sha256(json.dumps(content,sort_keys=True).encode()).hexdigest()\n\ndef require_compatible_recipe(saved, expected=None):\n    from training.data.common import recipe\n    try:\n        current=recipe(saved[\'classes\'],saved[\'seed\'],saved[\'block_size\'],saved[\'near_threshold\'])\n        current[\'groups_sha256\']=saved[\'groups_sha256\']\n        legacy = saved[\'version\']==1 and (saved[\'implementation_sha256\'],saved[\'reader_sha256\']) in LEGACY_SIGNATURES\n        if legacy:\n            comparable={k:v for k,v in saved.items() if k not in (\'version\',\'implementation_sha256\',\'reader_sha256\')}\n            if comparable!={k:v for k,v in current.items() if k not in (\'version\',\'implementation_sha256\',\'reader_sha256\')}:\n                raise ValueError(\'Legacy cache recipe parameters changed\')\n        elif saved!=current:\n            raise ValueError(\'Unsupported cache recipe/code; use a new dataset version\')\n        if expected is not None:\n            keys=(\'classes\',\'seed\',\'block_size\',\'near_threshold\',\'groups_sha256\')\n            if any(saved.get(k)!=expected.get(k) for k in keys):\n                raise ValueError(\'Cache recipe changed; choose a new cache version\')\n    except (KeyError,TypeError) as error:\n        raise ValueError(\'Invalid cache recipe\') from error\n\ndef dataset_content_fingerprint(manifest):\n    content={k:manifest[k] for k in (\'names\',\'source_zip_sha256\',\'recipe\')}\n    content[\'images\']=sorted(({k:r[k] for k in (\'sha256\',\'label_sha256\',\'group\',\'split\')}\n                              for r in manifest[\'images\']),key=lambda r:r[\'sha256\'])\n    return hashlib.sha256(json.dumps(content,sort_keys=True,separators=(\',\',\':\'),allow_nan=False).encode()).hexdigest()\n', 'training/data/__init__.py': '', 'training/experiments/__init__.py': '', 'training/data/common.py': "import hashlib, json\nfrom pathlib import Path\n\nCAR35 = [\n    'Speed_limit_40', 'Speed_limit_50', 'Speed_limit_60', 'Speed_limit_70', 'Speed_limit_80',\n    'No_entry', 'No_cars', 'No_cars_turn_left', 'No_cars_turn_right',\n    'No_turn_left', 'No_turn_right', 'No_U_turn', 'No_cars_U_turn',\n    'No_straight_ahead', 'No_turn_left_and_U_turn', 'No_turn_left_and_right',\n    'No_parking', 'No_stopping_and_parking', 'No_parking_odd_days', 'No_parking_even_days',\n    'Go_straight', 'Keep_right', 'Roundabout', 'One_way_road', 'Road_reserved_for_cars',\n    'Pedestrian_crossing', 'Children_crossing', 'Road_works', 'Slow_down',\n    'Dangerous_curve_left', 'Dangerous_curve_right',\n    'No_overtaking', 'End_no_overtaking', 'Give_way', 'End_all_prohibitions',\n]\nSPLITS = ('train', 'val', 'test')\nVERSION = 2\n\n\ndef digest_file(path):\n    with Path(path).open('rb') as stream:\n        digest = hashlib.sha256()\n        for block in iter(lambda: stream.read(1024*1024), b''): digest.update(block)\n        return digest.hexdigest()\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding='utf-8'))\n\n\ndef recipe(classes=CAR35, seed=42, block_size=100, near_threshold=4, groups=None):\n    from training.compatibility import implementation_digest\n    group_digest = hashlib.sha256(json.dumps(groups, sort_keys=True).encode()).hexdigest()\n    return dict(version=VERSION, classes=list(classes), seed=seed, block_size=block_size,\n        near_threshold=near_threshold, groups_sha256=group_digest,\n        implementation_sha256=implementation_digest(),\n        reader_sha256=hashlib.sha256(Path(__file__).with_name('audit.py').read_text(encoding='utf-8').encode()).hexdigest())\n", 'training/data/audit.py': '"""Read-only detection EDA, including files inside ZIPs. No training or remapping."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport random\nimport xml.etree.ElementTree as ET\nimport zipfile\n\nfrom PIL import Image\nimport yaml\n\nIMAGE_EXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.webp\', \'.tif\', \'.tiff\'}\n\n\nclass DatasetReader:\n    def __init__(self, root):\n        self.root = Path(root).resolve()\n        if self.root.is_file() and self.root.suffix.lower() == \'.zip\':\n            paths, base = [self.root], self.root.parent\n        elif self.root.is_dir():\n            paths, base = sorted(self.root.rglob(\'*\')), self.root\n        else:\n            raise FileNotFoundError(f\'Dataset directory or ZIP not found: {self.root}\')\n        self.entries, self.zips, self.warnings = {}, {}, []\n        for path in paths:\n            if not path.is_file() or path.is_symlink():\n                continue\n            name = path.relative_to(base).as_posix()\n            self.entries[name] = (path, None, path.stat().st_size)\n            if path.suffix.lower() == \'.zip\':\n                try:\n                    archive = zipfile.ZipFile(path)\n                    self.zips[name] = archive\n                    for item in archive.infolist():\n                        if not item.is_dir():\n                            self.entries[name + \'!/\' + item.filename] = (archive, item, item.file_size)\n                            if PurePosixPath(item.filename).suffix.lower() in {\'.zip\', \'.rar\', \'.7z\', \'.gz\', \'.tar\'}:\n                                self.warnings.append(f\'Nested archive not expanded: {name}!/{item.filename}\')\n                except (OSError, zipfile.BadZipFile) as error:\n                    self.warnings.append(f\'Cannot open archive {name}: {type(error).__name__}\')\n\n    def open(self, name):\n        source, member, _ = self.entries[name]\n        return source.open(member) if member is not None else source.open(\'rb\')\n\n    def read(self, name, max_bytes=256 * 1024 * 1024):\n        if self.entries[name][2] > max_bytes:\n            raise ValueError(f\'Annotation exceeds {max_bytes // 1024**2} MiB read limit\')\n        with self.open(name) as stream:\n            value = stream.read(max_bytes + 1)\n        if len(value) > max_bytes:\n            raise ValueError(\'Annotation read limit exceeded\')\n        return value\n\n    def close(self):\n        for archive in self.zips.values():\n            archive.close()\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self, *_):\n        self.close()\n\n\ndef split_name(name):\n    mapping = {\'train\': \'train\', \'training\': \'train\', \'val\': \'val\', \'valid\': \'val\',\n               \'validation\': \'val\', \'test\': \'test\', \'testing\': \'test\'}\n    found = {mapping[part.casefold()] for part in PurePosixPath(name).parts[:-1] if part.casefold() in mapping}\n    return next(iter(found)) if len(found) == 1 else \'unknown\'\n\n\ndef paired_files(name, extension, directory_pairs, case_names):\n    path = PurePosixPath(name)\n    candidates = {str(path.with_suffix(extension))}\n    parts = list(path.parts)\n    for i, part in enumerate(parts[:-1]):\n        for before, after in directory_pairs:\n            if part.casefold() == before.casefold():\n                changed = parts.copy(); changed[i] = after\n                candidates.add(str(PurePosixPath(*changed).with_suffix(extension)))\n    return sorted({actual for candidate in candidates for actual in case_names.get(candidate.casefold(), [])})\n\n\ndef audit_dataset(reader, dataset, image_limit=None, hash_limit=2000, seed=42,\n                  verify_decode=False, progress=None, annotation_format=\'auto\'):\n    """All counts refer to scanned images; partial/hash coverage is explicit.\n\n    Zero objects means empty_annotation only for a recognized annotation file.\n    Missing/unreadable/ambiguous labels never become verified background images.\n    """\n    if annotation_format not in {\'auto\', \'yolo\', \'voc\', \'coco\'}:\n        raise ValueError(\'annotation format must be auto, yolo, voc or coco\')\n    rng = random.Random(seed)\n    images = sorted(name for name in reader.entries if PurePosixPath(name).suffix.lower() in IMAGE_EXTENSIONS)\n    if (image_limit is not None and image_limit < 0) or (hash_limit is not None and hash_limit < 0):\n        raise ValueError(\'Limits must be nonnegative or None\')\n    selected = images if image_limit is None else sorted(rng.sample(images, min(image_limit, len(images))))\n    hashed = set(selected if hash_limit is None else rng.sample(selected, min(hash_limit, len(selected))))\n    issues, image_rows, boxes, catalogs = [], [], [], []\n    case_names = defaultdict(list)\n    for name in reader.entries:\n        case_names[name.casefold()].append(name)\n\n    def issue(path, message):\n        issues.append({\'dataset\': dataset, \'path\': path, \'message\': str(message)[:600]})\n\n    for warning in reader.warnings:\n        issue(\'\', warning)\n    for name in reader.entries:\n        suffix = PurePosixPath(name).suffix.lower()\n        try:\n            names = None\n            if suffix in {\'.yaml\', \'.yml\'}:\n                config = yaml.safe_load(reader.read(name, 1024**2))\n                if isinstance(config, dict) and \'names\' in config:\n                    names = config[\'names\']\n            elif PurePosixPath(name).name.casefold() in {\'classes.txt\', \'obj.names\', \'classes.names\'}:\n                names = reader.read(name, 1024**2).decode(\'utf-8-sig\').splitlines()\n            if names is not None:\n                if isinstance(names, list):\n                    names = dict(enumerate(names))\n                if not isinstance(names, dict) or not names:\n                    raise ValueError(\'Invalid class names\')\n                normalized = {int(key): str(value).strip() for key, value in names.items()}\n                if any(key < 0 or not value for key, value in normalized.items()):\n                    raise ValueError(\'Invalid class ID/name\')\n                catalogs.append((name, normalized))\n        except Exception as error:\n            issue(name, f\'Class metadata: {type(error).__name__}: {error}\')\n\n    def catalog_for(image_name):\n        matches = [(name, names) for name, names in catalogs\n                   if str(PurePosixPath(name).parent) == \'.\'\n                   or image_name.startswith(str(PurePosixPath(name).parent) + \'/\')]\n        if matches:\n            depth = max(len(PurePosixPath(name).parent.parts) for name, _ in matches)\n            closest = [(name, names) for name, names in matches if len(PurePosixPath(name).parent.parts) == depth]\n            if len({json.dumps(names, sort_keys=True) for _, names in closest}) > 1:\n                issue(image_name, \'Conflicting class maps at the same directory; class IDs remain unmapped.\')\n                return (\'ambiguous-class-map\', {})\n            return closest[0]\n        unique = {json.dumps(names, sort_keys=True) for _, names in catalogs}\n        return catalogs[0] if len(unique) == 1 else (\'unmapped\', {})\n\n    # COCO indexing: resolve full/suffix names; ambiguous duplicate basenames are errors.\n    by_basename = defaultdict(list)\n    for name in images:\n        by_basename[PurePosixPath(name).name].append(name)\n    coco = defaultdict(list)\n    for name in reader.entries:\n        if annotation_format not in {\'auto\', \'coco\'} or PurePosixPath(name).suffix.lower() != \'.json\':\n            continue\n        try:\n            content = json.loads(reader.read(name))\n            if not isinstance(content, dict) or not {\'images\', \'annotations\', \'categories\'} <= content.keys():\n                issue(name, \'JSON is not COCO detection; inspect separately (e.g. LabelMe/classification).\')\n                continue\n            categories = {row[\'id\']: str(row[\'name\']) for row in content[\'categories\']}\n            annotations = defaultdict(list)\n            for annotation in content[\'annotations\']:\n                annotations[annotation[\'image_id\']].append(annotation)\n            for image in content[\'images\']:\n                filename = str(image[\'file_name\']).replace(\'\\\\\', \'/\')\n                candidates = by_basename[PurePosixPath(filename).name]\n                suffix_matches = [key for key in candidates if key == filename or key.endswith(\'/\' + filename)]\n                if suffix_matches:\n                    candidates = suffix_matches\n                # Do not join a JSON in one archive to same-named images in another.\n                container = name.split(\'!/\', 1)[0] if \'!/\' in name else None\n                candidates = [key for key in candidates if\n                              (key.startswith(container + \'!/\') if container else \'!/\' not in key)]\n                if len(candidates) != 1:\n                    issue(name, f\'COCO image reference missing/ambiguous: {filename}\')\n                    continue\n                coco[candidates[0]].append((name, categories, image, annotations[image[\'id\']]))\n        except Exception as error:\n            issue(name, f\'JSON: {type(error).__name__}: {error}\')\n\n    iterator = progress(selected, desc=dataset, unit=\'image\') if progress else selected\n    for name in iterator:\n        row = {\'dataset\': dataset, \'path\': name, \'split\': split_name(name), \'parent_folder\': str(PurePosixPath(name).parent),\n               \'width\': None, \'height\': None, \'mode\': None, \'format\': None,\n               \'annotation_status\': \'missing_annotation\', \'boxes\': 0, \'sha256\': None}\n        image_rows.append(row)\n        try:\n            with reader.open(name) as stream, Image.open(stream) as image:\n                row.update(width=image.width, height=image.height, mode=image.mode)\n                if verify_decode:\n                    image.load()\n            if name in hashed:\n                digest = hashlib.sha256()\n                with reader.open(name) as stream:\n                    for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n                        digest.update(block)\n                row[\'sha256\'] = digest.hexdigest()\n        except Exception as error:\n            row[\'annotation_status\'] = \'unreadable_image\'\n            issue(name, f\'Image: {type(error).__name__}: {error}\')\n            continue\n        txt = paired_files(name, \'.txt\', [(\'images\', \'labels\')], case_names)\n        xml = paired_files(name, \'.xml\', [(\'JPEGImages\', \'Annotations\'), (\'images\', \'annotations\')], case_names)\n        options = [(\'yolo\', path) for path in txt] + [(\'voc\', path) for path in xml] + [(\'coco\', entry) for entry in coco[name]]\n        if annotation_format != \'auto\':\n            options = [option for option in options if option[0] == annotation_format]\n        if not options:\n            continue\n        if len(options) > 1:\n            row[\'annotation_status\'] = \'ambiguous_annotation\'\n            issue(name, \'Multiple matching annotations/formats; select a single export/root before training.\')\n            continue\n        kind, annotation = options[0]\n        row[\'format\'] = kind\n        raw_boxes, bad = [], False\n        try:\n            if kind == \'yolo\':\n                source, names = catalog_for(name)\n                for line_number, line in enumerate(reader.read(annotation, 8 * 1024**2).decode(\'utf-8-sig\').splitlines(), 1):\n                    if not line.strip():\n                        continue\n                    try:\n                        fields = line.split()\n                        if len(fields) != 5:\n                            raise ValueError(\'YOLO detection expects 5 columns; segmentation/other format is not inferred\')\n                        class_id = int(fields[0])\n                        if class_id < 0 or names and class_id not in names:\n                            raise ValueError(\'Class ID outside class map\')\n                        cx, cy, width, height = map(float, fields[1:])\n                        raw_boxes.append((str(class_id), names.get(class_id, f\'class_{class_id}\'), bool(names),\n                                          source, cx - width/2, cy - height/2, width, height, 0))\n                    except Exception as error:\n                        bad = True; issue(annotation, f\'Line {line_number}: {error}\')\n            elif kind == \'voc\':\n                tree = ET.fromstring(reader.read(annotation, 8 * 1024**2))\n                if tree.tag != \'annotation\':\n                    raise ValueError(\'Not a VOC annotation\')\n                for dimension in (\'width\', \'height\'):\n                    value = tree.findtext(\'size/\' + dimension)\n                    if value is not None and int(value) != row[dimension]:\n                        raise ValueError(\'VOC annotation size differs from image\')\n                for obj in tree.findall(\'object\'):\n                    label = obj.findtext(\'name\')\n                    if not label:\n                        raise ValueError(\'VOC object without class name\')\n                    xmin, ymin, xmax, ymax = [float(obj.findtext(\'bndbox/\' + key)) for key in (\'xmin\', \'ymin\', \'xmax\', \'ymax\')]\n                    raw_boxes.append((label, label, True, \'voc:names\', xmin/row[\'width\'], ymin/row[\'height\'],\n                                      (xmax-xmin)/row[\'width\'], (ymax-ymin)/row[\'height\'], 0))\n            else:\n                annotation_path, categories, image, objects = annotation\n                for dimension in (\'width\', \'height\'):\n                    if dimension in image and int(image[dimension]) != row[dimension]:\n                        raise ValueError(\'COCO annotation size differs from image\')\n                for obj in objects:\n                    class_id = obj[\'category_id\']\n                    if class_id not in categories:\n                        raise ValueError(\'COCO category missing from categories\')\n                    x, y, width, height = map(float, obj[\'bbox\'])\n                    raw_boxes.append((str(class_id), categories[class_id], True, annotation_path,\n                                      x/row[\'width\'], y/row[\'height\'], width/row[\'width\'], height/row[\'height\'], int(obj.get(\'iscrowd\', 0))))\n            for class_id, label, mapped, schema, x, y, width, height, crowd in raw_boxes:\n                if not all(math.isfinite(v) for v in (x, y, width, height)) or min(width, height) <= 0 or min(x, y) < -1e-6 or max(x+width, y+height) > 1+1e-6:\n                    bad = True; issue(name, \'Invalid bbox: nonfinite, empty, negative, or outside image\'); continue\n                boxes.append({\'dataset\': dataset, \'path\': name, \'split\': row[\'split\'], \'format\': kind,\n                              \'label_namespace\': schema, \'class_id\': class_id, \'class_name\': label, \'class_mapped\': mapped,\n                              \'x_norm\': x, \'y_norm\': y, \'width_norm\': width, \'height_norm\': height,\n                              \'area_fraction\': width * height, \'iscrowd\': crowd,\n                              \'short_side_at_640\': min(width*row[\'width\'], height*row[\'height\']) * 640 / max(row[\'width\'], row[\'height\'])})\n                row[\'boxes\'] += 1\n            row[\'annotation_status\'] = \'invalid_annotation\' if bad else \'annotated\' if row[\'boxes\'] else \'empty_annotation\'\n        except Exception as error:\n            row[\'annotation_status\'] = \'invalid_annotation\'\n            issue(name, f\'{kind}: {type(error).__name__}: {error}\')\n\n    summary = {\'dataset\': dataset, \'root\': str(reader.root), \'images_total\': len(images), \'images_scanned\': len(image_rows),\n               \'partial_scan\': len(selected) < len(images), \'images_hashed\': sum(bool(row[\'sha256\']) for row in image_rows),\n               \'hash_partial\': sum(bool(row[\'sha256\']) for row in image_rows) < len(images),\n               \'verify_decode\': verify_decode, \'annotation_format\': annotation_format,\n               \'valid_boxes\': len(boxes), \'issues\': len(issues),\n               \'invalid_annotations\': sum(row[\'annotation_status\'] == \'invalid_annotation\' for row in image_rows),\n               \'annotation_status\': dict(Counter(row[\'annotation_status\'] for row in image_rows)),\n               \'formats\': dict(Counter(row[\'format\'] for row in image_rows if row[\'format\'])),\n               \'file_extensions\': dict(Counter(PurePosixPath(name).suffix.lower() for name in reader.entries)),\n               \'class_maps\': [{\'path\': name, \'names\': names} for name, names in catalogs]}\n    return {\'summary\': summary, \'images\': image_rows, \'boxes\': boxes, \'issues\': issues}\n\n\ndef duplicate_groups(image_rows):\n    grouped = defaultdict(list)\n    for row in image_rows:\n        if row.get(\'sha256\'):\n            grouped[row[\'sha256\']].append(row)\n    return [{\'sha256\': digest, \'copies\': len(rows), \'cross_dataset\': len({r[\'dataset\'] for r in rows}) > 1,\n             \'cross_split\': len({r[\'split\'] for r in rows} - {\'unknown\'}) > 1,\n             \'members\': [{\'dataset\': r[\'dataset\'], \'path\': r[\'path\'], \'split\': r[\'split\']} for r in rows]}\n            for digest, rows in grouped.items() if len(rows) > 1]\n', 'training/data/cache.py': '"""Standalone SupSign subset preparation and YOLO11s experiments (no API changes)."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport random\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nfrom PIL import Image\nimport yaml\n\nfrom training.data.audit import DatasetReader, audit_dataset\n\nfrom training.data.common import *\nfrom training.compatibility import require_compatible_recipe, dataset_content_fingerprint\n\ndef verify_dataset(root):\n    root = Path(root).resolve();manifest = read_json(root/\'manifest.json\')\n    require_compatible_recipe(manifest[\'recipe\'])\n    config = yaml.safe_load((root/\'data.yaml\').read_text(encoding=\'utf-8\'))\n    if config != dict(path=\'.\',train=\'images/train\',val=\'images/val\',test=\'images/test\',names=manifest[\'names\']):\n        raise ValueError(\'Dataset YAML changed\')\n    hashes,groups,expected,labels = set(),{},set(),set()\n    for row in manifest[\'images\']:\n        image,label = root/row[\'image\'],root/row[\'label\'];split=row[\'split\']\n        if split not in SPLITS or image.parent != root/\'images\'/split or label != root/\'labels\'/split/(image.stem+\'.txt\'):\n            raise ValueError(\'Unsafe or misplaced manifest path\')\n        if row[\'sha256\'] in hashes or (row[\'group\'] in groups and groups[row[\'group\']]!=split):\n            raise ValueError(\'Duplicate image or cross-split capture group\')\n        if digest_file(image)!=row[\'sha256\'] or digest_file(label)!=row[\'label_sha256\']:raise ValueError(\'Dataset image/label changed\')\n        hashes.add(row[\'sha256\']);groups[row[\'group\']]=split;expected.add(image);labels.add(label)\n    if expected!=set((root/\'images\').rglob(\'*.*\')) or labels!=set((root/\'labels\').rglob(\'*.txt\')):\n        raise ValueError(\'Dataset inventory changed\')\n    if {r[\'split\'] for r in manifest[\'images\']}!=set(SPLITS):raise ValueError(\'Empty dataset split\')\n    return dict(images=len(expected),names=manifest[\'names\'],manifest_sha256=digest_file(root/\'manifest.json\'),\n                dataset_content_sha256=dataset_content_fingerprint(manifest),\n                independent_field_test=manifest[\'independent_field_test\'],split_provenance=manifest[\'split_provenance\'])\n\n\ndef summarize_prepared_dataset(root, *, imgsz=640):\n    """EDA of the exact prepared training set, without changing images/labels/splits."""\n    root = Path(root).resolve()\n    if type(imgsz) is not int or imgsz < 32:\n        raise ValueError(\'Invalid EDA input image size\')\n    manifest = read_json(root / \'manifest.json\')\n    names = manifest[\'names\']\n    images, boxes = [], []\n    counts = {split: Counter() for split in SPLITS}\n    negatives = 0\n    for row in manifest[\'images\']:\n        # Preparation/cache verification already checked hashes and decoded images.\n        with Image.open(root / row[\'image\']) as image:\n            width, height = image.size\n        entries = [line.split() for line in (root / row[\'label\']).read_text().splitlines() if line.strip()]\n        if not entries:\n            negatives += 1\n        images.append(dict(image=row[\'image\'], split=row[\'split\'], group=row[\'group\'],\n                           width=width, height=height, boxes=len(entries)))\n        for entry in entries:\n            cls = int(entry[0]); _, _, w, h = map(float, entry[1:])\n            counts[row[\'split\']][cls] += 1\n            scale = imgsz / max(width, height)\n            boxes.append(dict(image=row[\'image\'], split=row[\'split\'], class_id=cls, name=names[cls],\n                              width_px=w*width, height_px=h*height, area_fraction=w*h,\n                              aspect_ratio=w*width/(h*height),\n                              short_side_at_input=min(w*width,h*height)*scale))\n    catalogue = [dict(class_id=i, name=name, **{s: counts[s][i] for s in SPLITS})\n                 for i, name in enumerate(names)]\n    exclusions = Counter(row[\'reason\'] for row in manifest[\'excluded\'])\n    summary = dict(images=len(images), boxes=len(boxes), negative_images=negatives,\n                   groups=len({r[\'group\'] for r in images}), input_size=imgsz,\n                   boxes_under_16px=sum(b[\'short_side_at_input\'] < 16 for b in boxes),\n                   duplicates_removed=manifest[\'duplicates_removed\'], excluded_images=len(manifest[\'excluded\']),\n                   exclusion_reasons=dict(exclusions), missing_classes=manifest[\'missing_classes\'],\n                   split_provenance=manifest[\'split_provenance\'], independent_field_test=False,\n                   source_summary=manifest[\'source_summary\'], limitations=manifest[\'limitations\'])\n    return dict(summary=summary, class_counts=catalogue, images=images, boxes=boxes)\n\n\ndef pack_dataset(root,target):\n    root,target=Path(root).resolve(),Path(target).resolve()\n    if target.is_relative_to(root):raise ValueError(\'Dataset cache must be outside the dataset\')\n    if target.exists() or target.with_suffix(target.suffix+\'.sha256\').exists():raise FileExistsError(target)\n    verify_dataset(root);target.parent.mkdir(parents=True,exist_ok=True)\n    temporary=target.with_suffix(target.suffix+\'.part\')\n    if temporary.exists():raise FileExistsError(temporary)\n    try:\n        with zipfile.ZipFile(temporary,\'x\',compression=zipfile.ZIP_STORED) as archive:\n            for path in sorted(root.rglob(\'*\')):\n                if path.is_file():archive.write(path,path.relative_to(root).as_posix())\n        temporary.rename(target)\n        target.with_suffix(target.suffix+\'.sha256\').write_text(digest_file(target),encoding=\'utf-8\')\n    finally:\n        if temporary.exists():temporary.unlink()\n    return target\n\n\ndef restore_dataset(source,output,*,expected_recipe=None):\n    source,output=Path(source),Path(output).resolve()\n    if output.exists():raise FileExistsError(output)\n    if digest_file(source)!=source.with_suffix(source.suffix+\'.sha256\').read_text().strip():\n        raise ValueError(\'Dataset cache checksum mismatch\')\n    output.parent.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(source) as archive, tempfile.TemporaryDirectory(prefix=\'.restore-\',dir=output.parent) as temporary:\n        stage=Path(temporary)/\'dataset\';stage.mkdir();seen=set()\n        for member in archive.infolist():\n            relative=PurePosixPath(member.filename)\n            if (relative.is_absolute() or \'..\' in relative.parts or \'\\\\\' in member.filename\n                    or \':\' in member.filename or member.filename in seen\n                    or (member.external_attr>>16)&0o170000==0o120000):raise ValueError(\'Unsafe cache ZIP path\')\n            seen.add(member.filename)\n            if member.is_dir():continue\n            path=stage/relative;path.parent.mkdir(parents=True,exist_ok=True)\n            with archive.open(member) as stream,path.open(\'wb\') as target:shutil.copyfileobj(stream,target)\n        manifest=read_json(stage/\'manifest.json\')\n        require_compatible_recipe(manifest[\'recipe\'], expected_recipe)\n        verify_dataset(stage);stage.rename(output)\n    return manifest\n', 'training/data/downloads.py': '"""Download a shared Drive ZIP to local Colab storage, without account cookies."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nfrom urllib.parse import parse_qs, urlparse\nimport zipfile\n\n\nclass ZipChecksumMismatch(ValueError):\n    """Complete ZIP retained for diagnosis, never published as the expected file."""\n\n\ndef drive_file_id(source):\n    source = str(source).strip()\n    if not source or \'YOUR_\' in source or \'PASTE_\' in source:\n        raise ValueError(\'Paste a shared Google Drive file URL or file ID\')\n    if \'://\' in source:\n        parsed = urlparse(source)\n        if parsed.scheme != \'https\' or parsed.hostname != \'drive.google.com\' or parsed.username:\n            raise ValueError(\'Use a Google Drive file link, not a folder or another website\')\n        match = re.fullmatch(r\'/file/d/([A-Za-z0-9_-]+)(?:/.*)?\', parsed.path)\n        if match:\n            source = match[1]\n        elif parsed.path in (\'/uc\', \'/open\'):\n            source = parse_qs(parsed.query).get(\'id\', [\'\'])[0]\n        else:\n            raise ValueError(\'Use a Google Drive file link, not a folder\')\n    if not re.fullmatch(r\'[A-Za-z0-9_-]{10,}\', source):\n        raise ValueError(\'Invalid Google Drive file ID\')\n    return source\n\n\ndef download_drive_zip(source, target, *, expected_sha256=\'\', downloader=None):\n    """Publish only a complete ZIP; source receipts prevent resuming another file.\n\n    gdown owns its resumable temporary files. This helper keeps those on network\n    failure and never overwrites a pre-existing target from an unknown source.\n    An authoritative SHA-256 can adopt an already downloaded matching ZIP.\n    """\n    file_id = drive_file_id(source)\n    checksum = str(expected_sha256).strip().lower()\n    if checksum and not re.fullmatch(\'[a-f0-9]{64}\', checksum):\n        raise ValueError(\'Expected SHA-256 must have 64 hexadecimal characters\')\n    target = Path(target).resolve()\n    receipt = target.with_name(target.name+\'.source.json\')\n    staging = target.with_name(target.name+\'.download\')\n\n    def validate(path):\n        if not path.is_file() or not zipfile.is_zipfile(path):\n            raise ValueError(\'Download is not a ZIP. Check Drive sharing, quota and the file link.\')\n        if checksum:\n            with path.open(\'rb\') as stream:\n                actual = hashlib.file_digest(stream, \'sha256\').hexdigest()\n            if actual != checksum:\n                raise ZipChecksumMismatch(f\'Downloaded ZIP SHA-256 mismatch for {path.name}: \'\n                    f\'expected {checksum}; actual {actual}. Complete ZIP kept at {path}; check the source link/version.\')\n\n    known = receipt.exists()\n    if known:\n        try:\n            same_source = json.loads(receipt.read_text())[\'file_id\'] == file_id\n        except (ValueError, KeyError, TypeError):\n            same_source = False\n        if not same_source: raise FileExistsError(\'Local download belongs to another Drive file; choose a new path\')\n    elif target.exists():\n        if not checksum: raise FileExistsError(\'Existing ZIP has no source receipt; supply its SHA-256 or choose a new path\')\n        validate(target)\n    elif staging.exists() or any(target.parent.glob(staging.name+\'*.tmp\')):\n        raise FileExistsError(\'Unknown partial download exists; choose a new local path\')\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if not known:\n        with receipt.open(\'x\', encoding=\'utf-8\') as stream:\n            json.dump({\'file_id\': file_id}, stream)\n    if target.exists():\n        validate(target)\n        return target\n    if staging.exists():\n        try:\n            validate(staging)\n        except ZipChecksumMismatch:\n            raise\n        except ValueError:\n            staging.unlink()  # Only the staging file identified by our matching receipt.\n    if not staging.exists():\n        if downloader is None:\n            import gdown\n            downloader = gdown.download\n        try:\n            result = downloader(id=file_id, output=str(staging), quiet=False,\n                                resume=True, use_cookies=False)\n        except Exception as error:\n            raise RuntimeError(\'Drive download failed. Check public-link access/quota, then rerun this cell \'\n                               \'to resume in the same runtime.\') from error\n        if result is None:\n            raise RuntimeError(\'Drive returned no download. Check public-link access and download quota.\')\n    try:\n        validate(staging)\n    except ZipChecksumMismatch:\n        raise\n    except ValueError:\n        staging.unlink(missing_ok=True)\n        raise\n    if target.exists(): raise FileExistsError(\'Download target appeared; it was not overwritten\')\n    staging.rename(target)\n    return target\n', 'training/data/supsign.py': '"""Standalone SupSign subset preparation and YOLO11s experiments (no API changes)."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport random\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nfrom PIL import Image\nimport yaml\n\nfrom training.data.audit import DatasetReader, audit_dataset\n\nfrom training.data.common import *\ndef visual_hash(image):\n    grey = image.convert(\'L\').resize((9, 8), Image.Resampling.LANCZOS)\n    pixels = list(grey.tobytes()); result = 0\n    for y in range(8):\n        for x in range(8): result = (result << 1) | (pixels[y*9+x] > pixels[y*9+x+1])\n    return result\n\n\ndef cluster_records(rows, *, groups=None, block_size=100, near_threshold=4, progress=None):\n    """Union supplied captures, filename blocks and dHash neighbours transitively.\n\n    Filename blocks / visual hashes are only development heuristics, never proof of\n    independent trips or physical signs. Explicit groups replace filename blocks.\n    """\n    if block_size < 1 or (near_threshold is not None and not 0 <= near_threshold <= 8):\n        raise ValueError(\'Invalid grouping settings\')\n    parent = list(range(len(rows)))\n    def find(i):\n        while i != parent[i]: parent[i] = parent[parent[i]]; i = parent[i]\n        return i\n    def union(a, b):\n        a, b = find(a), find(b)\n        if a != b: parent[max(a,b)] = min(a,b)\n    seen = {}\n    tree = None\n    grouping = progress(rows, desc=\'Group related images\', unit=\'image\') if progress else rows\n    for index, row in enumerate(grouping):\n        for alias in row[\'aliases\']:\n            relative = alias.split(\'!/\', 1)[-1]\n            if groups is not None:\n                group = groups.get(relative, groups.get(alias))\n                if not isinstance(group, str) or not group.strip():\n                    raise ValueError(f\'Missing capture group: {relative}\')\n                key = (\'capture\', group.strip())\n            else:\n                match = re.fullmatch(r\'(.*?)([0-9]+)\', PurePosixPath(relative).stem)\n                key = (\'filename\', match[1], int(match[2]) // block_size) if match else (\'image\', row[\'sha256\'])\n            if key in seen: union(index, seen[key])\n            else: seen[key] = index\n        if near_threshold is None: continue\n        value = row[\'visual_hash\']\n        if tree is None: tree = (value, index, {}); continue\n        # BK tree lookup: no O(n^2) all-pairs hash matrix.\n        pending = [tree]\n        while pending:\n            other, representative, children = pending.pop()\n            distance = (value ^ other).bit_count()\n            if distance <= near_threshold: union(index, representative)\n            pending.extend(child for edge, child in children.items()\n                           if distance-near_threshold <= edge <= distance+near_threshold)\n        node = tree\n        while True:\n            distance = (value ^ node[0]).bit_count()\n            if distance == 0: break\n            if distance not in node[2]: node[2][distance] = (value, index, {}); break\n            node = node[2][distance]\n    return [f\'g{find(i):06d}\' for i in range(len(rows))]\n\n\ndef choose_splits(rows, names, seed):\n    grouped = defaultdict(list)\n    for row in rows: grouped[row[\'group\']].append(row)\n    if len(grouped) < 3: raise ValueError(\'Need at least 3 independent candidate groups; provide capture groups\')\n    keys = sorted(grouped)\n    group_counts = {key: Counter(int(line.split()[0]) for row in values for line in row[\'lines\'])\n                    for key, values in grouped.items()}\n    total = sum(group_counts.values(), Counter())\n    absent = [names[i] for i in range(len(names)) if not total[i]]\n    if absent: raise ValueError(\'No usable training annotations for classes: \'+str(absent))\n    support = {i: {key for key in keys if group_counts[key][i]} for i in range(len(names))}\n    forced_train = {next(iter(groups)) for groups in support.values() if len(groups) == 1}\n    eligible = [key for key in keys if key not in forced_train]\n    # Removing two holdout groups loses coverage only for a class supported by\n    # exactly those two groups. This finds a feasible seed pair without splitting\n    # a group or depending on forty random attempts to rescue a rare class.\n    incompatible = {frozenset(groups) for groups in support.values() if len(groups) == 2}\n    ratios = {\'train\': .7, \'val\': .15, \'test\': .15}\n    def balance(sizes):\n        return sum(abs(sizes[s]/len(rows)-ratios[s]) for s in SPLITS)\n    best = None\n    for trial in range(40):\n        rng = random.Random(seed+trial)\n        candidates = eligible.copy(); rng.shuffle(candidates)\n        pair = next(((a,b) for i,a in enumerate(candidates) for b in candidates[i+1:]\n                     if frozenset((a,b)) not in incompatible), None)\n        if pair is None:\n            raise ValueError(\'Cannot preserve train class coverage with non-empty val/test groups; \'\n                             \'provide more independent capture groups, or explicitly revise the taxonomy\')\n        assignments = dict.fromkeys(keys, \'train\')\n        sizes = Counter(train=len(rows))\n        counts = {\'train\': total.copy(), \'val\': Counter(), \'test\': Counter()}\n        for key, split in zip(pair, (\'val\',\'test\')):\n            assignments[key] = split\n            sizes[\'train\'] -= len(grouped[key]); sizes[split] += len(grouped[key])\n            counts[\'train\'].subtract(group_counts[key]); counts[split].update(group_counts[key])\n        order = keys.copy(); rng.shuffle(order)\n        order.sort(key=lambda k: -len(grouped[k]))\n        missing_count = sum(not counts[s][i] for s in (\'val\',\'test\') for i in range(len(names)))\n        score = missing_count*1e6 + balance(sizes)\n        for key in order:\n            gc = group_counts[key]\n            if assignments[key] != \'train\' or any(counts[\'train\'][c] <= n for c,n in gc.items()):\n                continue\n            choices = []\n            for split in (\'val\',\'test\'):\n                proposed = sizes.copy()\n                proposed[\'train\'] -= len(grouped[key]); proposed[split] += len(grouped[key])\n                remaining = missing_count-sum(not counts[split][c] for c in gc)\n                choices.append((remaining*1e6+balance(proposed), split, remaining, proposed))\n            cost, split, remaining, proposed = min(choices, key=lambda v: v[0])\n            if cost < score:\n                assignments[key] = split; sizes = proposed; missing_count = remaining; score = cost\n                counts[\'train\'].subtract(gc); counts[split].update(gc)\n        missing = {s: [i for i in range(len(names)) if not counts[s][i]] for s in SPLITS}\n        if best is None or score < best[0]: best = score, assignments, counts, missing\n    _, assignments, counts, missing = best\n    if missing[\'train\']: raise ValueError(\'Missing train classes after grouping: \'+str([names[i] for i in missing[\'train\']]))\n    for row in rows: row[\'split\'] = assignments[row[\'group\']]\n    return counts, {s: [names[i] for i in missing[s]] for s in SPLITS}\n\n\ndef prepare_supsign(source, output, *, classes=CAR35, groups=None, seed=42,\n                    block_size=100, near_threshold=4, progress=None):\n    source, output = Path(source).resolve(), Path(output).resolve()\n    if not source.is_file() or not zipfile.is_zipfile(source): raise ValueError(\'SOURCE_ZIP must be an existing YOLO ZIP\')\n    if output.exists(): raise FileExistsError(output)\n    if not classes or len(set(classes)) != len(classes): raise ValueError(\'Choose unique target classes\')\n    settings = recipe(classes, seed, block_size, near_threshold, groups)\n    excluded, rows, by_hash = [], [], defaultdict(list)\n    with DatasetReader(source) as reader:\n        if reader.warnings: raise ValueError(\'Source archive warnings: \'+str(reader.warnings[:3]))\n        report = audit_dataset(reader, \'supsign\', hash_limit=None, verify_decode=True,\n                               annotation_format=\'yolo\', progress=progress)\n        source_names = {name for catalog in report[\'summary\'][\'class_maps\'] for name in catalog[\'names\'].values()}\n        missing = set(classes)-source_names\n        if missing: raise ValueError(\'Missing source classes: \'+str(sorted(missing)))\n        boxes = defaultdict(list)\n        for box in report[\'boxes\']: boxes[box[\'path\']].append(box)\n        for image in report[\'images\']:\n            status = image[\'annotation_status\']\n            if status not in (\'annotated\',\'empty_annotation\'):\n                excluded.append(dict(source_path=image[\'path\'],reason=status));continue\n            if any(not b[\'class_mapped\'] for b in boxes[image[\'path\']]):\n                raise ValueError(\'Unmapped source class metadata\')\n            by_hash[image[\'sha256\']].append(image)\n        duplicates_removed = 0\n        candidates = sorted(by_hash.items())\n        if progress: candidates = progress(candidates, desc=\'Deduplicate / visual hashes\', unit=\'image\')\n        for digest, copies in candidates:\n            # Compare ALL source labels, not just selected classes. Disagreement is quarantined.\n            signatures = {tuple(sorted((b[\'class_name\'], *(round(b[k],8) for k in\n                          [\'x_norm\',\'y_norm\',\'width_norm\',\'height_norm\'])) for b in boxes[v[\'path\']])) for v in copies}\n            if len(signatures) != 1:\n                excluded.extend(dict(source_path=v[\'path\'],reason=\'conflicting_duplicate_labels\') for v in copies);continue\n            representative = min(copies,key=lambda v:v[\'path\']); lines = []\n            for b in boxes[representative[\'path\']]:\n                if b[\'class_name\'] not in classes:continue\n                x,y = b[\'x_norm\']+b[\'width_norm\']/2,b[\'y_norm\']+b[\'height_norm\']/2\n                lines.append(f"{classes.index(b[\'class_name\'])} {x:.10f} {y:.10f} {b[\'width_norm\']:.10f} {b[\'height_norm\']:.10f}")\n            with reader.open(representative[\'path\']) as stream, Image.open(stream) as image:\n                value = visual_hash(image) if near_threshold is not None else 0\n            rows.append(dict(source_path=representative[\'path\'],aliases=sorted(v[\'path\'] for v in copies),\n                             sha256=digest,lines=lines,visual_hash=value))\n            duplicates_removed += len(copies)-1\n        if not rows: raise ValueError(\'No usable images remain\')\n        for row, group in zip(rows,cluster_records(rows,groups=groups,block_size=block_size,\n                                                 near_threshold=near_threshold,progress=progress)):\n            row[\'group\'] = group\n        report[\'_split_provenance\'] = \'provided_capture_groups\' if groups is not None else \'development_heuristic\'\n        return finish_supsign_preparation(source, output, rows=rows, report=report,\n            excluded=excluded, duplicates_removed=duplicates_removed, settings=settings, progress=progress)\n\n\ndef finish_supsign_preparation(source, output, *, rows, report, excluded,\n                               duplicates_removed, settings, progress=None):\n    """Finish an audited/grouped snapshot; verify copied image hashes before publishing."""\n    from copy import deepcopy\n    source, output = Path(source).resolve(), Path(output).resolve()\n    if output.exists(): raise FileExistsError(output)\n    rows = deepcopy(rows)  # Keep failed-frame state available if finalization needs retry.\n    classes, seed = settings[\'classes\'], settings[\'seed\']\n    counts, missing = choose_splits(rows,classes,seed)\n    with DatasetReader(source) as reader:\n        if reader.warnings: raise ValueError(\'Source archive warnings: \'+str(reader.warnings[:3]))\n        output.parent.mkdir(parents=True,exist_ok=True)\n        with tempfile.TemporaryDirectory(prefix=\'.supsign-\',dir=output.parent) as temporary:\n            stage = Path(temporary)/\'dataset\';stage.mkdir()\n            writing = progress(rows, desc=\'Write prepared images\', unit=\'image\') if progress else rows\n            for row in writing:\n                suffix = PurePosixPath(row[\'source_path\']).suffix.lower()\n                row[\'image\'] = f"images/{row[\'split\']}/{row[\'sha256\']}{suffix}"\n                row[\'label\'] = f"labels/{row[\'split\']}/{row[\'sha256\']}.txt"\n                image, label = stage/row[\'image\'], stage/row[\'label\']\n                image.parent.mkdir(parents=True,exist_ok=True);label.parent.mkdir(parents=True,exist_ok=True)\n                with reader.open(row[\'source_path\']) as stream, image.open(\'wb\') as target:shutil.copyfileobj(stream,target)\n                if digest_file(image) != row[\'sha256\']:raise ValueError(\'Source image changed during preparation\')\n                label.write_text(\'\\n\'.join(row.pop(\'lines\'))+\'\\n\',encoding=\'utf-8\')\n                row[\'label_sha256\'] = digest_file(label);row.pop(\'visual_hash\')\n            manifest = dict(kind=\'supsign_car35_experiment\',recipe=settings,names=list(classes),\n                source_zip_sha256=digest_file(source),source_summary=report[\'summary\'],\n                images=rows,excluded=excluded,issues=report[\'issues\'],duplicates_removed=duplicates_removed,\n                class_counts={s:dict(counts[s]) for s in SPLITS},missing_classes=missing,\n                split_provenance=report.get(\'_split_provenance\', \'development_heuristic\'),\n                independent_field_test=False,\n                limitations=\'Exact-byte duplicates removed; dHash and filename blocks are heuristics. \'\n                            \'An independent car/windshield field test is still required. \'\n                            \'Source-negative/other-class images are negatives only for the selected taxonomy.\')\n            write_json(stage/\'manifest.json\',manifest)\n            config=dict(path=\'.\',train=\'images/train\',val=\'images/val\',test=\'images/test\',names=list(classes))\n            (stage/\'data.yaml\').write_text(yaml.safe_dump(config,sort_keys=False),encoding=\'utf-8\')\n            stage.rename(output)\n    return manifest\n\n\ndef recover_failed_preparation(traceback, *, progress=None):\n    """Resume the split failure in the current Python session, without re-auditing.\n\n    The traceback must retain prepare_supsign\'s fully audited and grouped rows.\n    Never accepts an arbitrary saved file as proof that the source was audited.\n    """\n    frame = None\n    while traceback is not None:\n        candidate = traceback.tb_frame\n        if (candidate.f_code.co_name == \'prepare_supsign\' and\n            Path(candidate.f_code.co_filename).name in (\'supsign_training.py\', \'supsign.py\')):\n            state = candidate.f_locals\n            if (state.get(\'rows\') and all(\'group\' in row and \'lines\' in row for row in state[\'rows\'])\n                and all(key in state for key in (\'source\',\'output\',\'report\',\'excluded\',\'duplicates_removed\'))):\n                frame = dict(state)\n        traceback = traceback.tb_next\n    if frame is None:\n        raise ValueError(\'No completed preparation snapshot in traceback; keep the original failed runtime \'\n                         \'or rerun preparation with the updated notebook\')\n    def inventory(reader):\n        return {name: (member.CRC, member.file_size) for name, (_, member, _) in reader.entries.items()\n                if member is not None}\n    original = frame.get(\'reader\')\n    if original is None: raise ValueError(\'Missing source inventory in preparation traceback\')\n    with DatasetReader(frame[\'source\']) as current:\n        if inventory(original) != inventory(current):\n            raise ValueError(\'Source ZIP changed since the failed preparation; rerun the audit\')\n    report = dict(frame[\'report\'])\n    report[\'_split_provenance\'] = \'provided_capture_groups\' if frame.get(\'groups\') is not None else \'development_heuristic\'\n    settings = recipe(frame[\'classes\'], frame[\'seed\'], frame[\'block_size\'],\n                      frame[\'near_threshold\'], frame.get(\'groups\'))\n    return finish_supsign_preparation(frame[\'source\'], frame[\'output\'], rows=frame[\'rows\'],\n        report=report, excluded=frame[\'excluded\'], duplicates_removed=frame[\'duplicates_removed\'],\n        settings=settings, progress=progress)\n\n\n\nfrom training.data.cache import verify_dataset, summarize_prepared_dataset, pack_dataset, restore_dataset\nfrom training.experiments.train import train_experiment\nfrom training.experiments.evaluate import evaluate_checkpoint\n', 'training/data/folder_dataset.py': '"""Prepare a downloaded YOLO folder as a versioned CAR35 development dataset.\n\nOnly data.yaml and train/valid/test image/label roots are read. Filename numbers\nare deliberately not treated as video/capture metadata. Legacy ZIP/cache recipes\nare unchanged; this artifact is consumed through its ordinary YOLO data.yaml.\n"""\nfrom collections import Counter, defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport tempfile\n\nfrom PIL import Image\nimport yaml\n\nfrom training.data.audit import DatasetReader, IMAGE_EXTENSIONS, audit_dataset\nfrom training.data.common import CAR35, SPLITS, digest_file, read_json, write_json\nfrom training.data.supsign import cluster_records, visual_hash\nfrom training.data.folder_splits import choose_folder_splits\n\n\nclass FolderReader(DatasetReader):\n    """Reuse the source auditor without recursively scanning a whole workspace."""\n    def __init__(self, root):\n        self.root = Path(root).resolve()\n        self.entries, self.zips, self.warnings = {}, {}, []\n        config = self.root/\'data.yaml\'\n        if not config.is_file():\n            raise FileNotFoundError(config)\n        paths = [config]\n        for split in (\'train\', \'valid\', \'test\'):\n            for kind in (\'images\', \'labels\'):\n                directory = self.root/split/kind\n                if not directory.is_dir():\n                    raise ValueError(f\'Missing source directory: {directory}\')\n                paths.extend(sorted(p for p in directory.rglob(\'*\') if p.is_file()))\n        for path in paths:\n            if path.is_symlink() or not path.resolve().is_relative_to(self.root):\n                raise ValueError(\'Source symlink resolves outside the dataset\')\n            name = path.relative_to(self.root).as_posix()\n            self.entries[name] = path, None, path.stat().st_size\n\n\ndef _audit(reader, workers, progress):\n    """Independent file batches; retain the existing label/geometry/decode audit."""\n    images = sorted(n for n in reader.entries if Path(n).suffix.lower() in IMAGE_EXTENSIONS)\n    if not images:\n        raise ValueError(\'No source images\')\n    # Use a bounded number of batches; files are only opened inside the workers.\n    batches = [images[start:start+512] for start in range(0, len(images), 512)]\n    annotations = {n: v for n, v in reader.entries.items() if Path(n).suffix.lower() not in IMAGE_EXTENSIONS}\n\n    def scan(names):\n        partial = object.__new__(DatasetReader)\n        partial.root, partial.zips, partial.warnings = reader.root, {}, []\n        # Include all labels to preserve the existing ambiguity checks.\n        partial.entries = {**annotations, **{n: reader.entries[n] for n in names}}\n        return audit_dataset(partial, \'downloaded_supsign\', hash_limit=None,\n                             verify_decode=True, annotation_format=\'yolo\')\n\n    reports = []\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        results = executor.map(scan, batches)\n        results = progress(results, desc=\'Audit source batches (512 images)\', unit=\'batch\') if progress else results\n        for report in results:\n            reports.append(report)\n    rows = [r for report in reports for r in report[\'images\']]\n    boxes = [b for report in reports for b in report[\'boxes\']]\n    issues = [i for report in reports for i in report[\'issues\']]\n    summary = dict(dataset=\'downloaded_supsign\', images_total=len(images), images_scanned=len(rows),\n                   images_hashed=sum(bool(r[\'sha256\']) for r in rows), partial_scan=False,\n                   hash_partial=False, verify_decode=True, annotation_format=\'yolo\', valid_boxes=len(boxes),\n                   annotation_status=dict(Counter(r[\'annotation_status\'] for r in rows)),\n                   issues=len(issues), class_maps=reports[0][\'summary\'][\'class_maps\'])\n    return dict(summary=summary, images=rows, boxes=boxes, issues=issues)\n\n\ndef prepare_folder_dataset(source, output, *, classes=CAR35, seed=42, near_threshold=4,\n                           progress=None, workers=8):\n    source, output = Path(source).resolve(), Path(output).resolve()\n    if output.exists():\n        raise FileExistsError(output)\n    inputs = [source/s for s in (\'train\', \'valid\', \'test\')]\n    if output == source or source.is_relative_to(output) or any(\n            output.is_relative_to(p) or p.is_relative_to(output) for p in inputs):\n        raise ValueError(\'Output overlaps source image/label roots\')\n    if not classes or len(set(classes)) != len(classes) or any(not isinstance(c,str) or not c for c in classes):\n        raise ValueError(\'Choose unique target class names\')\n    if type(workers) is not int or not 1 <= workers <= 32:\n        raise ValueError(\'workers must be 1..32\')\n    if near_threshold is not None and (type(near_threshold) is not int or not 0 <= near_threshold <= 8):\n        raise ValueError(\'Invalid dHash distance\')\n    implementation_files = {name:digest_file(Path(__file__).with_name(name)) for name in\n                            (\'folder_dataset.py\',\'folder_splits.py\',\'audit.py\',\'supsign.py\',\'common.py\')}\n    implementation_sha256 = hashlib.sha256(json.dumps(implementation_files,sort_keys=True).encode()).hexdigest()\n    source_yaml_sha256 = digest_file(source/\'data.yaml\')\n    with FolderReader(source) as reader:\n        report = _audit(reader, workers, progress)\n        available = {n for catalog in report[\'summary\'][\'class_maps\'] for n in catalog[\'names\'].values()}\n        if set(classes)-available:\n            raise ValueError(f\'Missing source classes: {sorted(set(classes)-available)}\')\n        by_path = defaultdict(list)\n        for box in report[\'boxes\']:\n            by_path[box[\'path\']].append(box)\n        copies, excluded = defaultdict(list), []\n        for image in report[\'images\']:\n            if image[\'annotation_status\'] not in (\'annotated\', \'empty_annotation\'):\n                excluded.append(dict(source_path=image[\'path\'], reason=image[\'annotation_status\']))\n                continue\n            if any(not b[\'class_mapped\'] for b in by_path[image[\'path\']]):\n                raise ValueError(\'Unmapped source classes\')\n            copies[image[\'sha256\']].append(image)\n        candidates, duplicates = [], 0\n\n        def convert(item):\n            digest, members = item\n            signatures = {tuple(sorted((b[\'class_name\'], *(round(b[k],8) for k in\n                           (\'x_norm\',\'y_norm\',\'width_norm\',\'height_norm\'))) for b in by_path[m[\'path\']]))\n                          for m in members}\n            if len(signatures) != 1:\n                return None, [dict(source_path=m[\'path\'],reason=\'conflicting_duplicate_labels\') for m in members], 0\n            representative = min(members,key=lambda m:m[\'path\'])\n            lines = []\n            for box in by_path[representative[\'path\']]:\n                if box[\'class_name\'] in classes:\n                    cx,cy = box[\'x_norm\']+box[\'width_norm\']/2, box[\'y_norm\']+box[\'height_norm\']/2\n                    lines.append(f"{classes.index(box[\'class_name\'])} {cx:.10f} {cy:.10f} {box[\'width_norm\']:.10f} {box[\'height_norm\']:.10f}")\n            with reader.open(representative[\'path\']) as stream, Image.open(stream) as image:\n                value = visual_hash(image) if near_threshold is not None else 0\n                width,height = image.size\n            return dict(source_path=representative[\'path\'], aliases=sorted(m[\'path\'] for m in members),\n                        sha256=digest, lines=lines, visual_hash=value, width=width, height=height), [], len(members)-1\n\n        with ThreadPoolExecutor(max_workers=workers) as executor:\n            results = executor.map(convert, sorted(copies.items()))\n            if progress: results=progress(results,desc=\'Deduplicate / visual hashes\',unit=\'image\')\n            for row, bad, removed in results:\n                excluded.extend(bad); duplicates += removed\n                if row is not None: candidates.append(row)\n        # No filename aliases are passed to the legacy clustering helper: only\n        # dHash edges may join distinct SHA-256 images in this new recipe.\n        grouping_rows = [dict(r,aliases=[]) for r in candidates]\n        groups = cluster_records(grouping_rows, near_threshold=near_threshold, progress=progress)\n        for row, group in zip(candidates, groups): row[\'group\']=group\n        if progress: print(\'Assigning whole groups to 70/15/15 target splits...\',flush=True)\n        counts, missing = choose_folder_splits(candidates,classes,seed)\n        negative_images = sum(not r[\'lines\'] for r in candidates)\n        output.parent.mkdir(parents=True,exist_ok=True)\n        with tempfile.TemporaryDirectory(prefix=\'.car35-folder-\',dir=output.parent) as temporary:\n            stage=Path(temporary)/\'dataset\';stage.mkdir()\n            for split in SPLITS:\n                (stage/\'images\'/split).mkdir(parents=True)\n                (stage/\'labels\'/split).mkdir(parents=True)\n\n            def copy(row):\n                row=dict(row)\n                suffix=Path(row[\'source_path\']).suffix.lower()\n                row[\'image\']=f"images/{row[\'split\']}/{row[\'sha256\']}{suffix}"\n                row[\'label\']=f"labels/{row[\'split\']}/{row[\'sha256\']}.txt"\n                shutil.copyfile(source/row[\'source_path\'],stage/row[\'image\'])\n                if digest_file(stage/row[\'image\']) != row[\'sha256\']:\n                    raise ValueError(\'Source image changed while copying\')\n                lines=row.pop(\'lines\')\n                (stage/row[\'label\']).write_text(\'\\n\'.join(lines)+ (\'\\n\' if lines else \'\'),encoding=\'utf-8\')\n                row[\'label_sha256\']=digest_file(stage/row[\'label\'])\n                row.pop(\'visual_hash\')\n                return row\n\n            with ThreadPoolExecutor(max_workers=workers) as executor:\n                results=executor.map(copy,candidates)\n                if progress: results=progress(results,desc=\'Write checked dataset copies\',unit=\'image\')\n                rows=list(results)\n            # With no explicit path, Ultralytics resolves paths beside the YAML.\n            # path=\'.\' would instead depend on the training process\'s cwd.\n            config=dict(train=\'images/train\',val=\'images/val\',test=\'images/test\',names=list(classes))\n            (stage/\'data.yaml\').write_text(yaml.safe_dump(config,sort_keys=False),encoding=\'utf-8\')\n            settings=dict(version=2,seed=seed,target_ratios=[.7,.15,.15],near_threshold=near_threshold,\n                          grouping=\'sha256_dedup_then_dhash_components\',filename_grouping=False,\n                          splitting=\'strongest_class_groups_train_then_ratio_balance\',\n                          implementation_files=implementation_files,\n                          implementation_sha256=implementation_sha256)\n            if digest_file(source/\'data.yaml\') != source_yaml_sha256:\n                raise ValueError(\'Source class mapping changed during preparation\')\n            manifest=dict(kind=\'car35_folder_development_v4\',names=list(classes),settings=settings,\n                          source_summary=report[\'summary\'],source_yaml_sha256=source_yaml_sha256,\n                          source_root=str(source),images=rows,excluded=excluded,issues=report[\'issues\'],\n                          duplicates_removed=duplicates,negative_images=negative_images,\n                          class_counts={s:dict(counts[s]) for s in SPLITS},missing_classes=missing,\n                          split_provenance=\'dhash_development_groups\',independent_field_test=False,\n                          limitations=\'dHash groups are development heuristics, not verified trips or physical signs. \'\n                          \'Negatives mean no selected CAR35 label, not no traffic signs. \'\n                          \'No guarantee against every near-duplicate or independent-field leakage. \'\n                          \'This is a new benchmark; do not directly compare its metrics with V3.\')\n            write_json(stage/\'manifest.json\',manifest)\n            _write_counts(stage,manifest)\n            stage.rename(output)\n    return manifest\n\n\ndef _write_counts(root, manifest):\n    with (root/\'class_counts.csv\').open(\'w\',newline=\'\',encoding=\'utf-8-sig\') as stream:\n        fields=[\'class_id\',\'name\',*SPLITS,\'total\']\n        writer=csv.DictWriter(stream,fieldnames=fields);writer.writeheader()\n        for i,name in enumerate(manifest[\'names\']):\n            counts={s:manifest[\'class_counts\'][s].get(i,manifest[\'class_counts\'][s].get(str(i),0)) for s in SPLITS}\n            writer.writerow(dict(class_id=i,name=name,**counts,total=sum(counts.values())))\n\n\ndef verify_folder_dataset(root, *, workers=8, progress=None):\n    root=Path(root).resolve();manifest=read_json(root/\'manifest.json\')\n    if type(workers) is not int or not 1 <= workers <= 32:\n        raise ValueError(\'workers must be 1..32\')\n    config=yaml.safe_load((root/\'data.yaml\').read_text(encoding=\'utf-8\'))\n    expected=dict(train=\'images/train\',val=\'images/val\',test=\'images/test\',names=manifest[\'names\'])\n    if config != expected: raise ValueError(\'Dataset YAML changed\')\n    images,labels,hashes,groups=set(),set(),set(),{}\n    counts={s:Counter() for s in SPLITS};split_images=Counter();group_sizes=Counter()\n    def check_row(row):\n        image,label=(root/row[k] for k in (\'image\',\'label\'));split=row[\'split\']\n        if (split not in SPLITS or not image.resolve().is_relative_to(root) or not label.resolve().is_relative_to(root)\n            or image.parent!=root/\'images\'/split or label!=root/\'labels\'/split/(image.stem+\'.txt\')):\n            raise ValueError(\'Unsafe dataset path\')\n        content=label.read_bytes()\n        if digest_file(image)!=row[\'sha256\'] or hashlib.sha256(content).hexdigest()!=row[\'label_sha256\']:\n            raise ValueError(\'Dataset image/label hash changed\')\n        current=Counter()\n        for line in content.decode(\'utf-8\').splitlines():\n            if not line.strip():continue\n            fields=line.split()\n            if len(fields)!=5: raise ValueError(\'Invalid label columns\')\n            cls=int(fields[0]);cx,cy,w,h=map(float,fields[1:])\n            if (not 0<=cls<len(manifest[\'names\']) or not all(math.isfinite(v) for v in (cx,cy,w,h))\n                or min(w,h)<=0 or min(cx-w/2,cy-h/2)<-1e-6 or max(cx+w/2,cy+h/2)>1+1e-6):\n                raise ValueError(\'Invalid class or bbox\')\n            current[cls]+=1\n        return row,image,label,current\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        results=executor.map(check_row,manifest[\'images\'])\n        if progress:results=progress(results,desc=\'Verify image/label hashes and boxes\',unit=\'image\')\n        for row,image,label,current in results:\n            split=row[\'split\']\n            if row[\'sha256\'] in hashes or (row[\'group\'] in groups and groups[row[\'group\']]!=split):\n                raise ValueError(\'Cross-split group or repeated image hash\')\n            counts[split].update(current)\n            images.add(image);labels.add(label);hashes.add(row[\'sha256\']);groups[row[\'group\']]=split\n            split_images[split]+=1;group_sizes[row[\'group\']]+=1\n    if images!=set((root/\'images\').rglob(\'*.*\')) or labels!=set((root/\'labels\').rglob(\'*.*\')):\n        raise ValueError(\'Dataset inventory changed\')\n    if set(split_images)!=set(SPLITS):raise ValueError(\'Empty dataset split\')\n    for split in SPLITS:\n        recorded={int(k):v for k,v in manifest[\'class_counts\'][split].items() if v}\n        if dict(counts[split])!=recorded:raise ValueError(\'Class counts changed\')\n    return dict(images=len(images),boxes=sum(sum(c.values()) for c in counts.values()),\n                names=manifest[\'names\'],split_images=dict(split_images),groups=len(groups),\n                split_groups=dict(Counter(groups.values())),largest_group_images=max(group_sizes.values()),\n                missing_classes=manifest[\'missing_classes\'],manifest_sha256=digest_file(root/\'manifest.json\'),\n                cross_split_exact_duplicates=0,cross_split_stored_groups=0,\n                independent_field_test=False)\n', 'training/data/folder_splits.py': '"""Group-level split balancing for a new development benchmark, not V3."""\nfrom collections import Counter, defaultdict\nimport random\n\nfrom training.data.common import SPLITS\n\n\ndef choose_folder_splits(rows, names, seed=42):\n    """Retain each class\'s strongest group in train, then balance whole groups.\n\n    The largest source of a rare class must not be moved wholesale to holdout\n    merely to make that class present there. Missing holdout coverage is reported.\n    Image ratios and mean per-class box ratios share equal objective weight;\n    missing holdout classes have a bounded penalty, never the V3 million-point\n    priority that could overwhelm the available learning support.\n    """\n    grouped=defaultdict(list)\n    for row in rows:grouped[row[\'group\']].append(row)\n    keys=sorted(grouped)\n    gc={k:Counter(int(line.split()[0]) for r in grouped[k] for line in r[\'lines\']) for k in keys}\n    total=sum(gc.values(),Counter())\n    absent=[names[c] for c in range(len(names)) if not total[c]]\n    if absent:raise ValueError(f\'No usable training annotations for classes: {absent}\')\n    anchors={min((k for k in keys if gc[k][c]),key=lambda k:(-gc[k][c],k)) for c in range(len(names))}\n    eligible=[k for k in keys if k not in anchors]\n    if len(eligible)<2:\n        raise ValueError(\'Cannot preserve train class coverage and strongest groups with non-empty val/test; provide more groups\')\n    ratios=dict(train=.7,val=.15,test=.15)\n    def image_cost(sizes):return sum(abs(sizes[s]/len(rows)-ratios[s]) for s in SPLITS)\n    def class_cost(counts):\n        return sum(abs(counts[s][c]/total[c]-ratios[s]) for c in range(len(names)) for s in SPLITS)/len(names)\n    best=None\n    for trial in range(8):\n        rng=random.Random(seed+trial)\n        assignments=dict.fromkeys(keys,\'train\')\n        sizes=Counter(train=len(rows));counts=dict(train=total.copy(),val=Counter(),test=Counter())\n        order=eligible.copy();rng.shuffle(order)\n        order.sort(key=lambda k:len(grouped[k]))\n        # Small seed groups avoid locking most of the dataset into one holdout.\n        for key,split in zip(order[:2],(\'val\',\'test\')):\n            assignments[key]=split;sizes[\'train\']-=len(grouped[key]);sizes[split]+=len(grouped[key])\n            counts[\'train\'].subtract(gc[key]);counts[split].update(gc[key])\n        missing=sum(not counts[s][c] for s in (\'val\',\'test\') for c in range(len(names)))\n        score=image_cost(sizes)+class_cost(counts)+.05*missing\n        # Scarce/large class contributions first, with seeded tie order.\n        rng.shuffle(order)\n        order.sort(key=lambda k:(-max((n/total[c] for c,n in gc[k].items()),default=0),-len(grouped[k])))\n        for key in order:\n            if assignments[key]!=\'train\':continue\n            choices=[]\n            for split in (\'val\',\'test\'):\n                proposed=sizes.copy();proposed[\'train\']-=len(grouped[key]);proposed[split]+=len(grouped[key])\n                delta=0\n                for c,n in gc[key].items():\n                    delta+=(abs((counts[\'train\'][c]-n)/total[c]-.7)-abs(counts[\'train\'][c]/total[c]-.7)\n                            +abs((counts[split][c]+n)/total[c]-ratios[split])-abs(counts[split][c]/total[c]-ratios[split]))/len(names)\n                filled=sum(not counts[split][c] for c in gc[key])\n                cost=score+image_cost(proposed)-image_cost(sizes)+delta-.05*filled\n                choices.append((cost,split,proposed,filled))\n            cost,split,proposed,filled=min(choices,key=lambda v:v[0])\n            if cost<score-1e-12:\n                assignments[key]=split;sizes=proposed;score=cost;missing-=filled\n                counts[\'train\'].subtract(gc[key]);counts[split].update(gc[key])\n        if best is None or score<best[0]:best=score,assignments,counts\n    _,assignments,counts=best\n    for row in rows:row[\'split\']=assignments[row[\'group\']]\n    return counts,{s:[name for c,name in enumerate(names) if not counts[s][c]] for s in SPLITS}\n', 'training/data/raw_car35.py': '"""Prepare a raw 107-class YOLO ZIP as a new CAR35 development benchmark."""\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nimport yaml\n\nfrom training.data.common import CAR35, digest_file, read_json, write_json\nfrom training.data.car35_bundle import verify_car35\nfrom training.data.folder_dataset import prepare_folder_dataset\n\n\ndef _recipe(dataset_version, seed, near_threshold):\n    if not isinstance(dataset_version, str) or not re.fullmatch(r\'[A-Za-z0-9][A-Za-z0-9_.-]{0,99}\', dataset_version):\n        raise ValueError(\'Supply a safe new dataset version\')\n    if type(seed) is not int or (near_threshold is not None and\n            (type(near_threshold) is not int or not 0 <= near_threshold <= 8)):\n        raise ValueError(\'Invalid preparation seed/dHash settings\')\n    files = {name:Path(__file__).with_name(name).read_text(encoding=\'utf-8\') for name in\n             (\'raw_car35.py\',\'folder_dataset.py\',\'folder_splits.py\',\'audit.py\',\'supsign.py\',\'common.py\')}\n    return dict(version=1, dataset_version=dataset_version, seed=seed, near_threshold=near_threshold,\n                implementation_sha256=hashlib.sha256(json.dumps(files,sort_keys=True).encode()).hexdigest())\n\n\ndef verify_raw_car35(root, *, dataset_version, seed=42, near_threshold=4, expected_source_sha256=\'\'):\n    """Reuse only a fully verified dataset from the same raw preparation recipe."""\n    manifest = read_json(Path(root)/\'manifest.json\')\n    raw = manifest.get(\'raw_zip_preparation\', {})\n    if raw.get(\'recipe\') != _recipe(dataset_version, seed, near_threshold):\n        raise ValueError(\'Raw CAR35 recipe changed; choose a new dataset version\')\n    if (raw.get(\'source_classes\') != 107 or\n            not re.fullmatch(\'[a-f0-9]{64}\', raw.get(\'source_zip_sha256\', \'\')) or\n            manifest.get(\'dataset_version\') != dataset_version):\n        raise ValueError(\'Invalid raw ZIP provenance\')\n    if expected_source_sha256 and raw[\'source_zip_sha256\'] != expected_source_sha256.lower():\n        raise ValueError(\'Raw source SHA-256 changed\')\n    if manifest[\'settings\'][\'seed\'] != seed or manifest[\'settings\'][\'near_threshold\'] != near_threshold:\n        raise ValueError(\'Stored split recipe differs\')\n    return verify_car35(root)\n\n\ndef _source_entries(archive):\n    entries, seen = archive.infolist(), set()\n    if len(entries) > 200000 or sum(m.file_size for m in entries) > 64*1024**3:\n        raise ValueError(\'Raw ZIP exceeds preparation limits\')\n    for member in entries:\n        name, path = member.filename, PurePosixPath(member.filename)\n        if (not name or path.is_absolute() or \'..\' in path.parts or \'\\\\\' in name or \':\' in name or\n                name.casefold() in seen or member.flag_bits & 1 or\n                (member.external_attr >> 16) & 0o170000 == 0o120000):\n            raise ValueError(\'Unsafe, encrypted or duplicate raw ZIP path\')\n        seen.add(name.casefold())\n    if \'data.yaml\' not in seen or archive.getinfo(\'data.yaml\').file_size > 1024**2:\n        raise ValueError(\'Raw ZIP must contain data.yaml at its root\')\n    config = yaml.safe_load(archive.read(\'data.yaml\'))\n    if not isinstance(config, dict): raise ValueError(\'Invalid raw data.yaml\')\n    names = config.get(\'names\')\n    if isinstance(names, list): names = dict(enumerate(names))\n    try:\n        normalized = {int(k):v for k,v in names.items()}\n    except (AttributeError, TypeError, ValueError) as error:\n        raise ValueError(\'Expected a 107-class source catalog\') from error\n    if (len(names) != 107 or set(normalized) != set(range(107)) or\n            config.get(\'nc\', 107) != 107 or any(not isinstance(v,str) or not v for v in normalized.values()) or\n            len(set(normalized.values())) != 107 or not set(CAR35) <= set(normalized.values())):\n        raise ValueError(\'Expected a raw 107-class catalog containing all CAR35 names\')\n    prefixes = tuple(f\'{split}/{kind}/\' for split in (\'train\',\'valid\',\'test\') for kind in (\'images\',\'labels\'))\n    for split in (\'train\',\'valid\',\'test\'):\n        for kind in (\'images\',\'labels\'):\n            if not any(not m.is_dir() and m.filename.startswith(f\'{split}/{kind}/\') for m in entries):\n                raise ValueError(f\'Missing raw source root: {split}/{kind}\')\n    expected = dict(train=\'train/images\', val=\'valid/images\', test=\'test/images\')\n    if any(config.get(k, v) != v for k,v in expected.items()):\n        raise ValueError(\'Raw YAML paths must use train/valid/test image roots\')\n    return [m for m in entries if not m.is_dir() and\n            (m.filename == \'data.yaml\' or m.filename.startswith(prefixes))]\n\n\ndef prepare_raw_car35_zip(source, output, *, dataset_version, seed=42, near_threshold=4,\n                          expected_source_sha256=\'\', workers=4, progress=None):\n    """Extract to an owned temporary tree, reuse the audited folder pipeline,\n    then publish a separately identified dataset. Never edits/deletes source ZIP.\n    The historical V4/R3 manifest and patch are not used for this new split.\n    """\n    source, output = Path(source).resolve(), Path(output).resolve()\n    if output.exists(): raise FileExistsError(output)\n    recipe = _recipe(dataset_version, seed, near_threshold)\n    if type(workers) is not int or not 1 <= workers <= 32: raise ValueError(\'workers must be 1..32\')\n    if expected_source_sha256 and not re.fullmatch(\'[a-fA-F0-9]{64}\', expected_source_sha256):\n        raise ValueError(\'Expected source SHA-256 must contain 64 hexadecimal characters\')\n    if progress: print(\'Checking raw ZIP SHA-256...\', flush=True)\n    actual = digest_file(source)\n    if expected_source_sha256 and actual != expected_source_sha256.lower():\n        raise ValueError(f\'Raw source SHA-256 mismatch: expected {expected_source_sha256}; actual {actual}\')\n    with zipfile.ZipFile(source) as archive:\n        entries = _source_entries(archive)\n        output.parent.mkdir(parents=True, exist_ok=True)\n        required = 2*sum(m.file_size for m in entries) + 1024**3\n        if shutil.disk_usage(output.parent).free < required:\n            raise ValueError(\'Not enough disk space for raw extraction and prepared CAR35\')\n        with tempfile.TemporaryDirectory(prefix=\'.raw-car35-\', dir=output.parent) as temporary:\n            stage = Path(temporary); raw = stage/\'source\'; raw.mkdir()\n            iterator = progress(entries, desc=\'Extract raw YOLO source\', unit=\'file\') if progress else entries\n            for member in iterator:\n                path = raw/member.filename; path.parent.mkdir(parents=True, exist_ok=True)\n                with archive.open(member) as reader, path.open(\'wb\') as writer:\n                    shutil.copyfileobj(reader, writer, length=1024*1024)\n            prepared = stage/\'prepared\'\n            manifest = prepare_folder_dataset(raw, prepared, classes=CAR35, seed=seed,\n                near_threshold=near_threshold, workers=workers, progress=progress)\n            # Keep repeat runs stable despite the randomly named extraction tree.\n            manifest[\'source_root\'] = \'raw_zip_sha256:\'+actual\n            manifest[\'dataset_version\'] = dataset_version\n            manifest[\'raw_zip_preparation\'] = dict(recipe=recipe, source_zip_sha256=actual, source_classes=107)\n            manifest[\'limitations\'] += \' Rebuilt from a raw 107-class ZIP; not the historical V4/R3 benchmark. No R3 overlay applied.\'\n            write_json(prepared/\'manifest.json\', manifest)\n            if progress: print(\'Verifying all prepared image/label hashes...\', flush=True)\n            info = verify_raw_car35(prepared, dataset_version=dataset_version, seed=seed,\n                near_threshold=near_threshold, expected_source_sha256=actual)\n            write_json(prepared/\'dataset_report.json\', info)\n            if digest_file(source) != actual: raise ValueError(\'Source ZIP changed during preparation\')\n            if output.exists(): raise FileExistsError(output)\n            prepared.rename(output)\n    return info\n', 'training/data/car35_bundle.py': '"""Restore/verify CAR35 V4 or a reviewed revision, without raw data or re-splitting."""\nfrom collections import Counter\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nimport yaml\nfrom training.data.common import CAR35, SPLITS, digest_file\n\n\ndef inspect_car35_archive(archive, *, expected_sha256, expected_manifest_sha256, allow_repacked=False):\n    """Preflight container identity only; restore still verifies every image/label.\n\n    A changed ZIP checksum can mean compression/timestamps changed. Accept that\n    only with explicit opt-in and the exact independently pinned manifest bytes.\n    Never learn a new trusted manifest from the downloaded archive itself.\n    """\n    archive = Path(archive)\n    for label, value in ((\'archive\', expected_sha256), (\'manifest\', expected_manifest_sha256)):\n        if not isinstance(value, str) or not re.fullmatch(\'[a-fA-F0-9]{64}\', value):\n            raise ValueError(f\'CAR35 {label} SHA-256 must be pinned (64 hexadecimal characters)\')\n    expected_sha256, expected_manifest_sha256 = expected_sha256.lower(), expected_manifest_sha256.lower()\n    actual = digest_file(archive)\n    repacked = actual != expected_sha256\n    if repacked and not allow_repacked:\n        raise ValueError(f\'CAR35 ZIP SHA-256 mismatch for {archive.name}: expected {expected_sha256}; actual {actual}\')\n    with zipfile.ZipFile(archive) as source:\n        entries = [m for m in source.infolist() if m.filename.casefold() == \'manifest.json\']\n        if len(entries) != 1 or entries[0].filename != \'manifest.json\' or entries[0].file_size > 64*1024**2:\n            raise ValueError(\'Expected a flat prepared CAR35 ZIP with one manifest.json at its root; check SOURCE_URL\')\n        manifest_sha = hashlib.sha256(source.read(entries[0])).hexdigest()\n    if manifest_sha != expected_manifest_sha256:\n        raise ValueError(f\'CAR35 manifest SHA-256 mismatch for {archive.name}: expected {expected_manifest_sha256}; \'\n                         f\'actual {manifest_sha}. Check SOURCE_URL: use the prepared V4 ZIP, not raw SupSign, \'\n                         \'a patch ZIP or a training-results ZIP.\')\n    return dict(archive_sha256=actual, expected_archive_sha256=expected_sha256,\n                manifest_sha256=manifest_sha, repacked=repacked)\n\n\ndef verify_car35(root, *, expected_manifest_sha256=\'\', workers=8):\n    root=Path(root).resolve();manifest_file=root/\'manifest.json\'\n    if expected_manifest_sha256 and digest_file(manifest_file)!=expected_manifest_sha256:\n        raise ValueError(\'CAR35 manifest SHA-256 mismatch\')\n    manifest=json.loads(manifest_file.read_text(encoding=\'utf-8\'))\n    if manifest.get(\'kind\') not in (\'car35_folder_development_v4\',\'car35_reviewed_development_v1\') or manifest.get(\'names\')!=CAR35:\n        raise ValueError(\'Expected prepared CAR35 V4 taxonomy, not raw SupSign or a V3 cache\')\n    if manifest[\'kind\']==\'car35_reviewed_development_v1\':\n        from training.data.car35_revision import validate_revision_provenance\n        validate_revision_provenance(root,manifest)\n    config=yaml.safe_load((root/\'data.yaml\').read_text(encoding=\'utf-8\'))\n    if config!=dict(train=\'images/train\',val=\'images/val\',test=\'images/test\',names=CAR35):\n        raise ValueError(\'Unsafe or changed CAR35 data.yaml\')\n    if manifest.get(\'independent_field_test\') is not False:\n        raise ValueError(\'This development bundle is not an independent field test\')\n    if type(workers) is not int or not 1<=workers<=32:raise ValueError(\'Invalid worker count\')\n    images,labels,hashes,groups=set(),set(),set(),{}\n    counts={s:Counter() for s in SPLITS};sizes=Counter()\n    def check(row):\n        split=row[\'split\'];image,label=(root/row[k] for k in (\'image\',\'label\'));sha=row[\'sha256\']\n        if (split not in SPLITS or not re.fullmatch(\'[0-9a-f]{64}\',sha)\n            or image.parent!=root/\'images\'/split or image.stem!=sha\n            or image.suffix.lower() not in {\'.png\',\'.jpg\',\'.jpeg\',\'.webp\',\'.bmp\',\'.tif\',\'.tiff\'}\n            or label!=root/\'labels\'/split/(sha+\'.txt\')\n            or not image.resolve().is_relative_to(root) or not label.resolve().is_relative_to(root)):\n            raise ValueError(\'Unsafe manifest image/label path\')\n        content=label.read_bytes()\n        if digest_file(image)!=sha or hashlib.sha256(content).hexdigest()!=row[\'label_sha256\']:\n            raise ValueError(\'CAR35 image/label hash changed\')\n        current=Counter()\n        for line in content.decode(\'utf-8\').splitlines():\n            if not line.strip():continue\n            fields=line.split()\n            if len(fields)!=5:raise ValueError(\'Invalid detection label\')\n            c=int(fields[0]);x,y,w,h=map(float,fields[1:])\n            if (not 0<=c<35 or not all(math.isfinite(v) for v in (x,y,w,h)) or min(w,h)<=0\n                or min(x-w/2,y-h/2)<-1e-6 or max(x+w/2,y+h/2)>1+1e-6):\n                raise ValueError(\'Invalid CAR35 class/bbox\')\n            current[c]+=1\n        return row,image,label,current\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        for row,image,label,current in pool.map(check,manifest[\'images\']):\n            if row[\'sha256\'] in hashes or (row[\'group\'] in groups and groups[row[\'group\']]!=row[\'split\']):\n                raise ValueError(\'Duplicate hash or cross-split CAR35 group\')\n            images.add(image);labels.add(label);hashes.add(row[\'sha256\']);groups[row[\'group\']]=row[\'split\']\n            sizes[row[\'split\']]+=1;counts[row[\'split\']].update(current)\n    actual_labels=set((root/\'labels\').rglob(\'*.*\'))\n    allowed_caches={root/\'labels\'/(s+suffix) for s in SPLITS for suffix in (\'.cache\',\'.cache.npy\')}\n    for cache in actual_labels & allowed_caches:\n        if cache.is_symlink() or not cache.is_file() or not cache.resolve().is_relative_to(root):\n            raise ValueError(\'Unsafe YOLO label cache path\')\n    if images!=set((root/\'images\').rglob(\'*.*\')) or labels!=actual_labels-allowed_caches:\n        raise ValueError(\'CAR35 image/label inventory changed\')\n    if set(sizes)!=set(SPLITS):raise ValueError(\'Empty CAR35 split\')\n    for split in SPLITS:\n        recorded={int(k):v for k,v in manifest[\'class_counts\'][split].items() if v}\n        if recorded!=dict(counts[split]):raise ValueError(\'CAR35 class counts changed\')\n    if any(not counts[\'train\'][c] for c in range(35)):raise ValueError(\'Missing CAR35 train class\')\n    return dict(kind=manifest[\'kind\'],dataset_version=manifest.get(\'dataset_version\',\'car35_v4\'),\n                images=len(images),boxes=sum(sum(c.values()) for c in counts.values()),names=list(CAR35),\n                split_images=dict(sizes),split_groups=dict(Counter(groups.values())),\n                class_counts={s:dict(counts[s]) for s in SPLITS},groups=len(groups),\n                manifest_sha256=digest_file(manifest_file),independent_field_test=False,\n                limitations=manifest.get(\'limitations\',\'Development split; scene overlap may remain.\'))\n\n\ndef restore_car35_bundle(archive,output,*,expected_sha256,expected_manifest_sha256=\'\'):\n    archive,output=Path(archive).resolve(),Path(output).resolve()\n    if output.exists():raise FileExistsError(output)\n    if not re.fullmatch(\'[0-9a-fA-F]{64}\',expected_sha256 or \'\'):\n        raise ValueError(\'CAR35 archive SHA-256 is required\')\n    if digest_file(archive)!=expected_sha256.lower():raise ValueError(\'CAR35 ZIP checksum mismatch\')\n    with zipfile.ZipFile(archive) as source:\n        entries=source.infolist();seen=set()\n        for member in entries:\n            name=member.filename;path=PurePosixPath(name)\n            if (not name or path.is_absolute() or \'..\' in path.parts or \'\\\\\' in name or \':\' in name\n                or name.casefold() in seen or (member.external_attr>>16)&0o170000==0o120000\n                or member.flag_bits&1):raise ValueError(\'Unsafe or duplicate CAR35 ZIP member\')\n            seen.add(name.casefold())\n        if not {\'data.yaml\',\'manifest.json\'}<=seen:raise ValueError(\'ZIP is not a flat prepared CAR35 bundle\')\n        required=sum(m.file_size for m in entries)+1024**3\n        output.parent.mkdir(parents=True,exist_ok=True)\n        if shutil.disk_usage(output.parent).free<required:raise ValueError(\'Not enough disk space to restore CAR35\')\n        with tempfile.TemporaryDirectory(prefix=\'.car35-restore-\',dir=output.parent) as temporary:\n            stage=Path(temporary)/\'dataset\';stage.mkdir()\n            for member in entries:\n                if member.is_dir():continue\n                target=stage/member.filename;target.parent.mkdir(parents=True,exist_ok=True)\n                with source.open(member) as stream,target.open(\'wb\') as dest:shutil.copyfileobj(stream,dest)\n            info=verify_car35(stage,expected_manifest_sha256=expected_manifest_sha256)\n            stage.rename(output)\n    return info\n', 'training/data/car35_revision.py': '"""Materialize reviewed labels from an immutable V4 ZIP into a new development dataset."""\nfrom collections import Counter\nfrom contextlib import contextmanager\nfrom copy import deepcopy\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nfrom training.data.common import CAR35, SPLITS, digest_file, write_json\nfrom training.data.car35_bundle import restore_car35_bundle, verify_car35\nfrom training.data.label_patch import create_label_patch\n\n\nREVISION_KIND = \'car35_reviewed_development_v1\'\n\n\ndef _sha(value):\n    if not isinstance(value, str) or not re.fullmatch(\'[a-f0-9]{64}\', value):\n        raise ValueError(\'Invalid revision SHA-256\')\n    return value\n\n\ndef _version(value):\n    if not isinstance(value, str) or not re.fullmatch(r\'[A-Za-z0-9][A-Za-z0-9_.-]{0,100}\', value):\n        raise ValueError(\'Supply a safe, separately identified dataset version\')\n    return value\n\n\ndef revision_manifest(source, patch, *, source_sha, patch_document_sha, patch_archive_sha, dataset_version):\n    """Deterministic identity; preserve every source image, group and split."""\n    _version(dataset_version)\n    for value in (source_sha, patch_document_sha, patch_archive_sha): _sha(value)\n    if (source.get(\'kind\') != \'car35_folder_development_v4\' or source.get(\'names\') != CAR35 or\n            source.get(\'independent_field_test\') is not False or\n            patch.get(\'kind\') != \'car35_reviewed_label_patch\' or patch.get(\'schema_version\') != 2 or\n            patch.get(\'source_manifest_sha256\') != source_sha or patch.get(\'names\') != CAR35 or\n            patch.get(\'split_changed\') is not False or patch.get(\'independent_field_test\') is not False):\n        raise ValueError(\'Invalid revision source/patch provenance\')\n    result = deepcopy(source)\n    rows = {r[\'sha256\']:r for r in result[\'images\']}\n    counts = {s:Counter({int(k):v for k,v in source[\'class_counts\'][s].items()}) for s in SPLITS}\n    seen, edited, removed, changed = set(), set(), 0, 0\n    changes = patch.get(\'changes\', [])\n    if not changes or len(changes) != patch.get(\'changed_images\'):\n        raise ValueError(\'Invalid patch image inventory\')\n    for item in changes:\n        sha = item[\'image_sha256\'];row = rows.get(sha)\n        if (row is None or sha in seen or item[\'split\'] != row[\'split\'] or item[\'label\'] != row[\'label\'] or\n                item[\'old_sha256\'] != row[\'label_sha256\']):\n            raise ValueError(\'Patch differs from source revision inventory\')\n        seen.add(sha);row[\'label_sha256\'] = _sha(item[\'new_sha256\'])\n        if not item.get(\'edits\'): raise ValueError(\'Patch has no reviewed edits\')\n        for edit in item[\'edits\']:\n            index = edit.get(\'line_index\');before,after = edit.get(\'from_class\'),edit.get(\'to_class\')\n            action = edit.get(\'action\')\n            if (edit.get(\'image_sha256\') != sha or edit.get(\'reviewed\') is not True or\n                    type(index) is not int or index < 0 or (sha,index) in edited or\n                    any(type(c) is not int or not 0 <= c < 35 for c in (before,after)) or\n                    action not in (\'replace_class\',\'remove_duplicate\') or\n                    not isinstance(edit.get(\'reason\'),str) or not edit[\'reason\'].strip()):\n                raise ValueError(\'Invalid reviewed revision edit\')\n            edited.add((sha,index));counts[row[\'split\']][before] -= 1\n            if action == \'replace_class\':\n                counts[row[\'split\']][after] += 1;changed += 1\n            else:\n                removed += 1\n    if (patch.get(\'removed_boxes\') != removed or patch.get(\'relabeled_boxes\') != changed or\n            patch.get(\'changed_boxes\') != removed+changed or\n            any(v < 0 for c in counts.values() for v in c.values())):\n        raise ValueError(\'Patch revision counts are inconsistent\')\n    result.update(kind=REVISION_KIND, dataset_version=dataset_version,\n        class_counts={s:{str(c):n for c,n in sorted(counts[s].items()) if n} for s in SPLITS},\n        missing_classes={s:[name for c,name in enumerate(CAR35) if not counts[s][c]] for s in SPLITS},\n        label_revision=dict(schema_version=1, source_manifest_sha256=source_sha,\n            patch_document_sha256=patch_document_sha, patch_archive_sha256=patch_archive_sha,\n            changed_images=len(changes), relabeled_boxes=changed, removed_boxes=removed,\n            split_changed=False, validation_labels_changed=any(c[\'split\']==\'val\' for c in changes),\n            test_labels_changed=any(c[\'split\']==\'test\' for c in changes)),\n        limitations=source.get(\'limitations\',\'Development split; scene overlap may remain.\')+\n            \' Reviewed label revision; image/group/split membership is unchanged. \'\n            \'Changed validation/test labels change the evaluation reference. Not an independent field test.\')\n    return result\n\n\ndef validate_revision_provenance(root, manifest):\n    root = Path(root)\n    original, patch = root/\'provenance/v4_manifest.json\', root/\'provenance/label_patch.json\'\n    if any(p.is_symlink() or not p.resolve().is_relative_to(root.resolve()) for p in (original,patch)):\n        raise ValueError(\'Unsafe revision provenance path\')\n    revision = manifest[\'label_revision\']\n    if (digest_file(original) != revision[\'source_manifest_sha256\'] or\n            digest_file(patch) != revision[\'patch_document_sha256\']):\n        raise ValueError(\'Revision provenance hash changed\')\n    expected = revision_manifest(json.loads(original.read_text(encoding=\'utf-8\')),\n        json.loads(patch.read_text(encoding=\'utf-8\')), source_sha=revision[\'source_manifest_sha256\'],\n        patch_document_sha=revision[\'patch_document_sha256\'], patch_archive_sha=revision[\'patch_archive_sha256\'],\n        dataset_version=manifest[\'dataset_version\'])\n    if expected != manifest:\n        raise ValueError(\'Revision manifest differs from source/patch provenance\')\n\n\n@contextmanager\ndef _patch_directory(archive, expected_sha256, parent=None):\n    archive = Path(archive).resolve()\n    if digest_file(archive) != _sha(expected_sha256): raise ValueError(\'Patch ZIP checksum mismatch\')\n    with zipfile.ZipFile(archive) as source:\n        entries = source.infolist();seen = set()\n        if len(entries) > 10000 or sum(m.file_size for m in entries) > 64*1024**2:\n            raise ValueError(\'Patch ZIP exceeds labels-only size limit\')\n        for member in entries:\n            name = member.filename;relative = PurePosixPath(name)\n            if (not name or relative.is_absolute() or \'..\' in relative.parts or \'\\\\\' in name or \':\' in name or\n                    name.casefold() in seen or member.flag_bits&1 or\n                    (member.external_attr>>16)&0o170000 == 0o120000):\n                raise ValueError(\'Unsafe or duplicate patch ZIP path\')\n            seen.add(name.casefold())\n        if \'patch.json\' not in seen or source.getinfo(\'patch.json\').file_size > 1024**2:\n            raise ValueError(\'Missing or oversized patch.json\')\n        with tempfile.TemporaryDirectory(prefix=\'.car35-patch-\',dir=parent) as temporary:\n            root = Path(temporary)\n            for member in entries:\n                if member.is_dir(): continue\n                target = root/member.filename;target.parent.mkdir(parents=True,exist_ok=True)\n                with source.open(member) as stream,target.open(\'wb\') as dest: shutil.copyfileobj(stream,dest)\n            yield root\n\n\ndef _prepare_revision(root, patch_root, *, source_sha, patch_sha, dataset_version):\n    raw = (root/\'manifest.json\').read_bytes()\n    if hashlib.sha256(raw).hexdigest() != _sha(source_sha): raise ValueError(\'Source manifest SHA-256 mismatch\')\n    patch_raw = (patch_root/\'patch.json\').read_bytes();patch = json.loads(patch_raw)\n    with tempfile.TemporaryDirectory(prefix=\'.replay-patch-\',dir=patch_root.parent) as temporary:\n        replay_root = Path(temporary)/\'replayed\'\n        edits = [edit for item in patch[\'changes\'] for edit in item[\'edits\']]\n        replay = create_label_patch(root,replay_root,edits,expected_manifest_sha256=source_sha)\n        if replay != patch: raise ValueError(\'Patch metadata differs from reviewed edit replay\')\n        expected_labels = {item[\'label\'] for item in replay[\'changes\']}\n        if {p.relative_to(patch_root).as_posix() for p in (patch_root/\'labels\').rglob(\'*\') if p.is_file()} != expected_labels:\n            raise ValueError(\'Patch label inventory differs from reviewed edits\')\n        for name in expected_labels:\n            if (replay_root/name).read_bytes() != (patch_root/name).read_bytes():\n                raise ValueError(\'Patch label bytes differ from reviewed edit replay\')\n    manifest = revision_manifest(json.loads(raw),patch,source_sha=source_sha,\n        patch_document_sha=hashlib.sha256(patch_raw).hexdigest(), patch_archive_sha=patch_sha,\n        dataset_version=dataset_version)\n    manifest_bytes = json.dumps(manifest,ensure_ascii=False,indent=2,allow_nan=False).encode(\'utf-8\')\n    return manifest,manifest_bytes,raw,patch_raw\n\n\ndef preview_car35_revision(root, patch_archive, *, expected_source_manifest_sha256,\n                           expected_patch_sha256, dataset_version):\n    """Verify source/patch read-only and predict the exact new manifest checksum."""\n    root = Path(root).resolve();_version(dataset_version)\n    verify_car35(root,expected_manifest_sha256=expected_source_manifest_sha256)\n    with _patch_directory(patch_archive,expected_patch_sha256) as patch_root:\n        manifest,raw,_,_ = _prepare_revision(root,patch_root,source_sha=expected_source_manifest_sha256,\n            patch_sha=expected_patch_sha256,dataset_version=dataset_version)\n    return dict(manifest=manifest,manifest_sha256=hashlib.sha256(raw).hexdigest(),\n        images=len(manifest[\'images\']),boxes=sum(sum(c.values()) for c in manifest[\'class_counts\'].values()),\n        dataset_version=dataset_version,materialized=False)\n\n\ndef restore_car35_revision(archive, patch_archive, output, *, expected_sha256,\n        expected_source_manifest_sha256, expected_patch_sha256, dataset_version, expected_manifest_sha256=\'\'):\n    """Restore once into owned staging, replay reviewed edits, verify, then publish."""\n    output = Path(output).resolve();_version(dataset_version)\n    if output.exists(): raise FileExistsError(output)\n    output.parent.mkdir(parents=True,exist_ok=True)\n    with _patch_directory(patch_archive,expected_patch_sha256,output.parent) as patch_root:\n        with tempfile.TemporaryDirectory(prefix=\'.car35-revision-\',dir=output.parent) as temporary:\n            stage = Path(temporary)/\'dataset\'\n            restore_car35_bundle(archive,stage,expected_sha256=expected_sha256,\n                                 expected_manifest_sha256=expected_source_manifest_sha256)\n            manifest,raw,original,patch_raw = _prepare_revision(stage,patch_root,\n                source_sha=expected_source_manifest_sha256,patch_sha=expected_patch_sha256,dataset_version=dataset_version)\n            if expected_manifest_sha256 and hashlib.sha256(raw).hexdigest()!=expected_manifest_sha256:\n                raise ValueError(\'Revised manifest SHA-256 mismatch\')\n            provenance = stage/\'provenance\';provenance.mkdir()\n            (provenance/\'v4_manifest.json\').write_bytes(original)\n            (provenance/\'label_patch.json\').write_bytes(patch_raw)\n            # Keep historical previews/reports explicitly separate from revised labels.\n            historical = provenance/\'v4_reports\';historical.mkdir()\n            for name in (\'README.md\',\'report.html\',\'dataset_report.json\',\'previews\'):\n                if (stage/name).exists(): (stage/name).rename(historical/name)\n            for item in json.loads(patch_raw)[\'changes\']:\n                shutil.copyfile(patch_root/item[\'label\'],stage/item[\'label\'])\n            for split in SPLITS:\n                for suffix in (\'.cache\',\'.cache.npy\'): (stage/\'labels\'/(split+suffix)).unlink(missing_ok=True)\n            (stage/\'manifest.json\').write_bytes(raw)\n            with (stage/\'class_counts.csv\').open(\'w\',newline=\'\',encoding=\'utf-8-sig\') as stream:\n                writer=csv.DictWriter(stream,fieldnames=[\'class_id\',\'name\',*SPLITS]);writer.writeheader()\n                for i,name in enumerate(CAR35):\n                    writer.writerow(dict(class_id=i,name=name,**{s:manifest[\'class_counts\'][s].get(str(i),0) for s in SPLITS}))\n            (stage/\'README.md\').write_text(\'# \'+dataset_version+\'\\n\\nReviewed label revision of CAR35 V4. \'\n                \'Images/groups/splits are unchanged; historical reports are under provenance/v4_reports. \'\n                \'Changed holdout labels change the evaluation reference. Not an independent field test.\\n\',encoding=\'utf-8\')\n            info=verify_car35(stage,expected_manifest_sha256=hashlib.sha256(raw).hexdigest())\n            write_json(stage/\'dataset_report.json\',info)\n            if output.exists(): raise FileExistsError(output)\n            stage.rename(output)\n    return info\n\n\ndef pack_car35_revision(root, target):\n    root,target = Path(root).resolve(),Path(target).resolve()\n    if target.is_relative_to(root) or root.is_relative_to(target): raise ValueError(\'ZIP must be outside dataset\')\n    sidecar=target.with_suffix(target.suffix+\'.sha256\')\n    if target.exists() or sidecar.exists(): raise FileExistsError(target)\n    info=verify_car35(root)\n    if info[\'kind\'] != REVISION_KIND: raise ValueError(\'Expected a reviewed CAR35 revision\')\n    caches={root/\'labels\'/(s+suffix) for s in SPLITS for suffix in (\'.cache\',\'.cache.npy\')}\n    paths=sorted(p for p in root.rglob(\'*\') if p.is_file() and p not in caches)\n    if any(p.is_symlink() or not p.resolve().is_relative_to(root) for p in paths): raise ValueError(\'Unsafe revision file\')\n    target.parent.mkdir(parents=True,exist_ok=True)\n    if shutil.disk_usage(target.parent).free < sum(p.stat().st_size for p in paths)+1024**3:\n        raise ValueError(\'Not enough disk space to pack CAR35 revision\')\n    with tempfile.TemporaryDirectory(prefix=\'.pack-revision-\',dir=target.parent) as temporary:\n        archive=Path(temporary)/\'dataset.zip\'\n        with zipfile.ZipFile(archive,\'x\',compression=zipfile.ZIP_STORED,allowZip64=True) as dest:\n            for path in paths: dest.write(path,path.relative_to(root).as_posix())\n        checksum=digest_file(archive)\n        if target.exists(): raise FileExistsError(target)\n        archive.rename(target)\n    sidecar.write_text(checksum+\'\\n\',encoding=\'utf-8\')\n    return dict(info,zip_sha256=checksum)\n', 'training/data/label_patch.py': '"""Export small, reviewed CAR35 label overlays; never mutate the source dataset."""\nfrom collections import defaultdict\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport re\nimport tempfile\n\nfrom training.data.common import CAR35, digest_file, write_json\n\n\ndef create_label_patch(root, output, edits, *, expected_manifest_sha256):\n    """Hash-bound label replacements, without images or a standalone training YAML.\n\n    line_index refers to a physical, zero-based source label line. Replace its\n    class token, or remove a reviewed duplicate with an unchanged retained box.\n    Bounding-box text, whitespace and line endings of other labels remain intact.\n    An applying tool must create a new dataset identity and verify\n    the source hashes; copying these labels into V4 in place is not supported.\n    """\n    root, output = Path(root).resolve(), Path(output).resolve()\n    if output.is_relative_to(root) or root.is_relative_to(output):\n        raise ValueError(\'Patch output must be outside the source dataset\')\n    if output.exists():\n        raise FileExistsError(output)\n    manifest_file = root/\'manifest.json\'\n    if (not re.fullmatch(\'[a-f0-9]{64}\', expected_manifest_sha256 or \'\') or\n            digest_file(manifest_file) != expected_manifest_sha256):\n        raise ValueError(\'Source manifest changed or checksum is invalid\')\n    manifest = json.loads(manifest_file.read_text(encoding=\'utf-8\'))\n    if manifest.get(\'kind\') != \'car35_folder_development_v4\' or manifest.get(\'names\') != CAR35:\n        raise ValueError(\'Expected the original CAR35 V4 manifest and class order\')\n    rows = {r[\'sha256\']: r for r in manifest[\'images\']}\n    if len(rows) != len(manifest[\'images\']):\n        raise ValueError(\'Duplicate source image identity\')\n    if not edits:\n        raise ValueError(\'Supply at least one reviewed edit\')\n    grouped, seen = defaultdict(list), set()\n    for edit in edits:\n        sha, index = edit.get(\'image_sha256\', \'\'), edit.get(\'line_index\')\n        before, after = edit.get(\'from_class\'), edit.get(\'to_class\')\n        action = edit.get(\'action\', \'replace_class\')\n        if (not re.fullmatch(\'[a-f0-9]{64}\', sha) or sha not in rows or\n                type(index) is not int or index < 0 or edit.get(\'reviewed\') is not True or\n                any(type(c) is not int or not 0 <= c < 35 for c in (before, after)) or\n                action not in (\'replace_class\', \'remove_duplicate\') or\n                action == \'replace_class\' and before == after or\n                not isinstance(edit.get(\'reason\'), str) or not edit[\'reason\'].strip()):\n            raise ValueError(\'Invalid, unreviewed or unsupported CAR35 edit\')\n        if (sha, index) in seen:\n            raise ValueError(\'Duplicate edit for one label line\')\n        seen.add((sha, index)); grouped[sha].append(dict(edit, action=action))\n    changed, content_by_path = [], {}\n    for sha, changes in sorted(grouped.items()):\n        row = rows[sha]; split = row[\'split\']\n        if split not in (\'train\', \'val\', \'test\'):\n            raise ValueError(\'Invalid source split\')\n        image, label = root/row[\'image\'], root/row[\'label\']\n        if (image.parent != root/\'images\'/split or image.stem != sha or\n                row[\'label\'] != f\'labels/{split}/{sha}.txt\' or\n                not image.resolve().is_relative_to(root) or not label.resolve().is_relative_to(root)):\n            raise ValueError(\'Unsafe source image/label path\')\n        raw = label.read_bytes()\n        if hashlib.sha256(raw).hexdigest() != row[\'label_sha256\']:\n            raise ValueError(\'Source label changed\')\n        if digest_file(image) != sha:\n            raise ValueError(\'Source image changed\')\n        lines = raw.decode(\'utf-8\').splitlines(keepends=True)\n        original_lines, removed = lines.copy(), set()\n        for edit in changes:\n            index = edit[\'line_index\']\n            if index >= len(lines):\n                raise ValueError(\'Label line no longer exists\')\n            fields = original_lines[index].split()\n            if len(fields) != 5 or fields[0] != str(edit[\'from_class\']):\n                raise ValueError(\'Source class/label line differs from reviewed edit\')\n            x, y, w, h = map(float, fields[1:])\n            if (not all(math.isfinite(v) for v in (x, y, w, h)) or min(w, h) <= 0 or\n                    min(x-w/2, y-h/2) < -1e-6 or max(x+w/2, y+h/2) > 1+1e-6):\n                raise ValueError(\'Source bbox requires separate review\')\n            if edit[\'action\'] == \'remove_duplicate\':\n                retained = edit.get(\'retained_line_index\')\n                if (type(retained) is not int or not 0 <= retained < len(original_lines) or\n                        retained == index or (sha, retained) in seen):\n                    raise ValueError(\'Duplicate removal requires an unchanged retained label line\')\n                other = original_lines[retained].split()\n                if (len(other) != 5 or other[0] != str(edit[\'to_class\']) or\n                        tuple(map(float, other[1:])) != (x, y, w, h)):\n                    raise ValueError(\'Retained duplicate must have the reviewed class and identical bbox\')\n                removed.add(index)\n            else:\n                lines[index] = re.sub(r\'^(\\s*)\\d+\', lambda m: m[1]+str(edit[\'to_class\']), lines[index], count=1)\n        updated = \'\'.join(line for i, line in enumerate(lines) if i not in removed).encode(\'utf-8\')\n        content_by_path[row[\'label\']] = updated\n        changed.append(dict(image_sha256=sha, split=split, label=row[\'label\'],\n                            old_sha256=row[\'label_sha256\'], new_sha256=hashlib.sha256(updated).hexdigest(),\n                            edits=changes))\n    removed_boxes = sum(e[\'action\'] == \'remove_duplicate\' for changes in grouped.values() for e in changes)\n    result = dict(schema_version=2, kind=\'car35_reviewed_label_patch\', source_manifest_sha256=expected_manifest_sha256,\n                  names=list(CAR35), changed_images=len(changed), changed_boxes=len(edits), changes=changed,\n                  removed_boxes=removed_boxes, relabeled_boxes=len(edits)-removed_boxes,\n                  standalone_training_dataset=False, independent_field_test=False,\n                  source_modified=False, split_changed=False,\n                  usage=\'Review overlay only. Materialize a separately identified dataset on a writable copy before training; \'\n                        \'preserve the original V4 ZIP/directory and update manifest hashes/counts. Never apply into V4 in place.\')\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(prefix=\'.label-patch-\', dir=output.parent) as temporary:\n        stage = Path(temporary)/\'patch\'; stage.mkdir()\n        for relative, content in content_by_path.items():\n            target = stage/PurePosixPath(relative); target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(content)\n        write_json(stage/\'patch.json\', result)\n        with (stage/\'corrections.csv\').open(\'w\', newline=\'\', encoding=\'utf-8-sig\') as stream:\n            fields = [\'image_sha256\', \'split\', \'action\', \'line_index\', \'retained_line_index\', \'from_class\', \'from_name\', \'to_class\', \'to_name\', \'reason\']\n            writer = csv.DictWriter(stream, fieldnames=fields); writer.writeheader()\n            for row in changed:\n                for edit in row[\'edits\']:\n                    writer.writerow(dict(image_sha256=row[\'image_sha256\'], split=row[\'split\'],\n                                         action=edit[\'action\'], retained_line_index=edit.get(\'retained_line_index\', \'\'),\n                                         line_index=edit[\'line_index\'], from_class=edit[\'from_class\'],\n                                         from_name=CAR35[edit[\'from_class\']], to_class=edit[\'to_class\'],\n                                         to_name=CAR35[edit[\'to_class\']], reason=edit[\'reason\']))\n        (stage/\'README.md\').write_text(\'# CAR35 reviewed label overlay\\n\\n\'+result[\'usage\']+\'\\n\', encoding=\'utf-8\')\n        if output.exists():\n            raise FileExistsError(output)\n        stage.rename(output)\n    return result\n', 'training/experiments/car35.py': '"""Train/export prepared CAR35 development data; never changes application serving."""\nimport json\nimport math\nfrom pathlib import Path\nimport zipfile\n\nfrom training.data.car35_bundle import verify_car35\nfrom training.data.common import digest_file, read_json, write_json\n\n\ndef train_car35(root,run_dir,*,epochs=80,imgsz=640,batch=8,device=\'0\',resume=False,yolo_factory=None):\n    root,run_dir=Path(root).resolve(),Path(run_dir).resolve()\n    if run_dir.is_relative_to(root) or root.is_relative_to(run_dir):raise ValueError(\'Run must be outside dataset\')\n    if epochs<1 or imgsz<32 or batch<1:raise ValueError(\'Invalid training settings\')\n    dataset=verify_car35(root)\n    settings=dict(model=\'yolo11s.pt\',epochs=epochs,imgsz=imgsz,batch=batch,seed=42)\n    request=run_dir/\'training_request.json\'\n    if resume:\n        if not request.is_file() or not (run_dir/\'weights/last.pt\').is_file():\n            raise ValueError(\'Resume requires last.pt and training_request.json in this runtime\')\n        previous=read_json(request)\n        if previous[\'dataset\'][\'manifest_sha256\']!=dataset[\'manifest_sha256\']:raise ValueError(\'Resume dataset changed\')\n        if previous[\'settings\']!=settings:raise ValueError(\'Resume settings changed\')\n        weights=run_dir/\'weights/last.pt\'\n    else:\n        if run_dir.exists():raise FileExistsError(run_dir)\n        weights=\'yolo11s.pt\'\n    if yolo_factory is None:\n        from ultralytics import YOLO\n        yolo_factory=YOLO\n    detector=yolo_factory(str(weights))\n    if resume:\n        names=detector.names;names=[names[i] for i in range(len(names))] if isinstance(names,dict) else list(names)\n        if detector.task!=\'detect\' or names!=dataset[\'names\']:raise ValueError(\'Resume checkpoint taxonomy differs\')\n    if not resume:\n        run_dir.mkdir(parents=True)\n        write_json(request,dict(dataset=dataset,settings=settings,test_evaluated=False))\n    options=dict(data=str(root/\'data.yaml\'),epochs=epochs,imgsz=imgsz,batch=batch,device=device,\n        project=str(run_dir.parent),name=run_dir.name,exist_ok=True,split=\'val\',workers=2,\n        seed=42,deterministic=True,cache=False,amp=True,patience=20,save=True,save_period=5,\n        fliplr=0.,flipud=0.,degrees=0.,shear=0.,perspective=0.,mixup=0.,mosaic=.5,\n        close_mosaic=min(10,epochs),plots=True)\n    if resume:options[\'resume\']=True\n    detector.train(**options)\n    best=run_dir/\'weights/best.pt\'\n    if not best.is_file():raise ValueError(\'Training did not produce best.pt\')\n    return best\n\n\ndef evaluate_car35(checkpoint,root,output,*,split=\'val\',allow_test=False,imgsz=640,batch=8,device=\'0\',yolo_factory=None):\n    if split not in (\'val\',\'test\') or split==\'test\' and not allow_test:raise ValueError(\'Held-out test requires explicit opt-in\')\n    root,output,checkpoint=Path(root).resolve(),Path(output).resolve(),Path(checkpoint).resolve()\n    if output.exists():raise FileExistsError(output)\n    if output.is_relative_to(root) or root.is_relative_to(output):raise ValueError(\'Evaluation must be outside dataset\')\n    dataset=verify_car35(root)\n    if yolo_factory is None:\n        from ultralytics import YOLO\n        yolo_factory=YOLO\n    detector=yolo_factory(str(checkpoint));names=detector.names\n    names=[names[i] for i in range(len(names))] if isinstance(names,dict) else list(names)\n    if detector.task!=\'detect\' or names!=dataset[\'names\']:raise ValueError(\'Evaluation checkpoint taxonomy differs\')\n    output.mkdir(parents=True)\n    result=detector.val(data=str(root/\'data.yaml\'),split=split,imgsz=imgsz,batch=batch,device=device,\n                        project=str(output.parent),name=output.name,exist_ok=True,plots=True)\n    report=dict(dataset=dataset,split=split,split_images=dataset[\'split_images\'][split],\n                checkpoint_sha256=digest_file(checkpoint),independent_field_test=False,\n                metrics={k:float(v) if math.isfinite(float(v)) else None for k,v in result.results_dict.items()})\n    box=getattr(result,\'box\',None)\n    if box is not None:\n        report[\'per_class\']=[dict(class_id=int(c),name=names[int(c)],precision=float(box.p[j]),\n            recall=float(box.r[j]),map50=float(box.ap50[j]),map50_95=float(box.ap[j])) for j,c in enumerate(box.ap_class_index)]\n    write_json(output/\'evaluation_report.json\',report)\n    return report\n\n\ndef export_car35_run(run_dir,root,target):\n    run_dir,root,target=Path(run_dir).resolve(),Path(root).resolve(),Path(target).resolve()\n    if target.exists():raise FileExistsError(target)\n    if target.is_relative_to(run_dir) or target.is_relative_to(root):raise ValueError(\'Export must be outside run/dataset\')\n    if not any((run_dir/\'weights\'/n).is_file() for n in (\'best.pt\',\'last.pt\')):raise ValueError(\'No checkpoint to export yet\')\n    dataset=verify_car35(root)\n    request=read_json(run_dir/\'training_request.json\')\n    if request[\'dataset\'][\'manifest_sha256\']!=dataset[\'manifest_sha256\']:raise ValueError(\'Export dataset changed\')\n    handoff=dict(schema_version=1,dataset=dataset,settings=request[\'settings\'],independent_field_test=False,\n        checkpoints={n:digest_file(run_dir/\'weights\'/n) for n in (\'best.pt\',\'last.pt\') if (run_dir/\'weights\'/n).is_file()},\n        limitations=dataset[\'limitations\'])\n    target.parent.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(target,\'x\',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:\n        for path in sorted(run_dir.rglob(\'*\')):\n            if path.is_file() and not path.is_symlink():archive.write(path,\'run/\'+path.relative_to(run_dir).as_posix())\n        for name in (\'manifest.json\',\'data.yaml\',\'dataset_report.json\',\'class_counts.csv\',\'README.md\'):\n            if (root/name).is_file():archive.write(root/name,\'dataset/\'+name)\n        for name in (\'v4_manifest.json\',\'label_patch.json\'):\n            if (root/\'provenance\'/name).is_file():archive.write(root/\'provenance\'/name,\'dataset/provenance/\'+name)\n        archive.writestr(\'handoff.json\',json.dumps(handoff,ensure_ascii=False,indent=2))\n    return target\n', 'training/experiments/train.py': '"""Standalone SupSign subset preparation and YOLO11s experiments (no API changes)."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport random\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nfrom PIL import Image\nimport yaml\n\nfrom training.data.audit import DatasetReader, audit_dataset\n\nfrom training.data.common import *\nfrom training.data.cache import verify_dataset\n\ndef train_experiment(root,run_dir,*,epochs=80,imgsz=640,batch=8,device=\'0\',resume=False,yolo_factory=None):\n    root,run_dir=Path(root).resolve(),Path(run_dir).resolve()\n    if run_dir.is_relative_to(root):raise ValueError(\'Training run must be outside dataset\')\n    if epochs<1 or imgsz<32 or batch<1:raise ValueError(\'Invalid training settings\')\n    dataset=verify_dataset(root)\n    request_path=run_dir/\'training_request.json\'\n    settings=dict(model=\'yolo11s.pt\',epochs=epochs,imgsz=imgsz,batch=batch)\n    if resume:\n        if not (run_dir/\'weights/last.pt\').is_file() or not request_path.is_file():raise ValueError(\'Resume requires saved last.pt and training_request.json\')\n        previous=read_json(request_path)\n        if previous[\'manifest_sha256\']!=dataset[\'manifest_sha256\']:raise ValueError(\'Resume dataset changed\')\n        if previous[\'settings\']!=settings:raise ValueError(\'Resume settings changed; use the original settings or a new run\')\n        model=run_dir/\'weights/last.pt\'\n    else:\n        if run_dir.exists():raise FileExistsError(\'Run exists; set RESUME=True or choose another RUN_NAME\')\n        model=\'yolo11s.pt\'\n    if yolo_factory is None:\n        from ultralytics import YOLO\n        yolo_factory=YOLO\n    detector=yolo_factory(str(model))\n    if not resume:\n        run_dir.mkdir(parents=True)\n        write_json(request_path,dict(**dataset,settings=settings,test_evaluated=False))\n    # Dataset archive remains portable; runtime absolute path lives beside the run.\n    config=yaml.safe_load((root/\'data.yaml\').read_text(encoding=\'utf-8\'));config[\'path\']=str(root)\n    runtime_data=run_dir/\'data_runtime.yaml\';runtime_data.write_text(yaml.safe_dump(config,sort_keys=False),encoding=\'utf-8\')\n    options=dict(data=str(runtime_data),epochs=epochs,imgsz=imgsz,batch=batch,device=device,\n                 project=str(run_dir.parent),name=run_dir.name,exist_ok=True,workers=2,\n                 seed=42,deterministic=True,cache=False,amp=True,patience=20,save=True,save_period=5,\n                 fliplr=0.,flipud=0.,degrees=0.,shear=0.,perspective=0.,mixup=0.,\n                 mosaic=.5,close_mosaic=min(10,epochs),plots=True)\n    if resume:options[\'resume\']=True\n    detector.train(**options)\n    if not (run_dir/\'weights/best.pt\').is_file():raise ValueError(\'Training did not produce best.pt\')\n    return run_dir/\'weights/best.pt\'\n', 'training/experiments/evaluate.py': '"""Standalone SupSign subset preparation and YOLO11s experiments (no API changes)."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport random\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nfrom PIL import Image\nimport yaml\n\nfrom training.data.audit import DatasetReader, audit_dataset\n\nfrom training.data.common import *\nfrom training.data.cache import verify_dataset\n\ndef evaluate_checkpoint(checkpoint,root,output,*,split=\'val\',allow_test=False,imgsz=640,batch=8,yolo_factory=None):\n    checkpoint,root,output=Path(checkpoint),Path(root).resolve(),Path(output).resolve()\n    if split not in (\'val\',\'test\') or split==\'test\' and not allow_test:raise ValueError(\'Held-out test must be explicitly enabled after choosing settings\')\n    if output.exists():raise FileExistsError(output)\n    if output.is_relative_to(root):raise ValueError(\'Evaluation must be outside dataset\')\n    dataset=verify_dataset(root)\n    if yolo_factory is None:\n        from ultralytics import YOLO\n        yolo_factory=YOLO\n    detector=yolo_factory(str(checkpoint))\n    names=detector.names\n    names=[names[i] for i in range(len(names))] if isinstance(names,dict) else list(names)\n    if detector.task!=\'detect\' or names!=dataset[\'names\']:raise ValueError(\'Checkpoint taxonomy differs from dataset\')\n    output.mkdir(parents=True)\n    config=yaml.safe_load((root/\'data.yaml\').read_text());config[\'path\']=str(root)\n    runtime_data=output/\'data_runtime.yaml\';runtime_data.write_text(yaml.safe_dump(config),encoding=\'utf-8\')\n    result=detector.val(data=str(runtime_data),split=split,imgsz=imgsz,batch=batch,\n                        project=str(output.parent),name=output.name,exist_ok=True,plots=True)\n    report=dict(**dataset,split=split,checkpoint_sha256=digest_file(checkpoint),\n                metrics={k:float(v) if math.isfinite(float(v)) else None for k,v in result.results_dict.items()})\n    box=getattr(result,\'box\',None)\n    if box is not None:\n        report[\'per_class\']=[dict(class_id=int(i),name=names[int(i)],precision=float(box.p[j]),\n                                 recall=float(box.r[j]),map50=float(box.ap50[j]),map50_95=float(box.ap[j]))\n                             for j,i in enumerate(box.ap_class_index)]\n        report[\'classes_without_gt\']=[n for i,n in enumerate(names) if i not in set(box.ap_class_index)]\n    write_json(output/\'evaluation_report.json\',report)\n    return report\n'}
PIPELINE_ID = hashlib.sha256(json.dumps(TRAINING_FILES, sort_keys=True).encode()).hexdigest()
CODE_ROOT = WORK / 'pipeline' / PIPELINE_ID
for relative, source in TRAINING_FILES.items():
    target = CODE_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
for name in list(sys.modules):
    if name in ('training','training.compatibility') or name.startswith(('training.data','training.experiments')):
        del sys.modules[name]
sys.path.insert(0, str(CODE_ROOT))
importlib.invalidate_caches()
from training.data.common import CAR35
from training.data.downloads import download_drive_zip, drive_file_id
from training.data.raw_car35 import prepare_raw_car35_zip, verify_raw_car35
from training.experiments.car35 import train_car35, evaluate_car35, export_car35_run
print('Pipeline raw107:', PIPELINE_ID[:16])


## 2. Chuẩn bị CAR35 từ ZIP raw 107 lớp

Lần đầu cần tải/giải nén, decode/hash ảnh, đọc nhãn và gom gần trùng nên có thể lâu.
Có thanh tiến độ từng giai đoạn; không in hàng chục nghìn đường dẫn.
Tên số thứ tự không được dùng làm capture group. Nhóm gần trùng nằm trong cùng split.
Tỷ lệ thực tế có thể khác 70/15/15 do kích thước nhóm; lớp hiếm thiếu holdout sẽ được báo.
Dataset đã xác minh trong cùng runtime được dùng lại; không tải/audit lại raw ZIP.


In [ ]:
from tqdm.auto import tqdm
progress = lambda items, desc, unit: tqdm(items, desc=desc, unit=unit)
if DATASET_ROOT.exists():
    print('Đang kiểm tra hash của CAR35 local...', flush=True)
    dataset_info = verify_raw_car35(DATASET_ROOT, dataset_version=DATASET_VERSION, seed=SEED,
        near_threshold=NEAR_HASH_DISTANCE, expected_source_sha256=RAW_ZIP_SHA256)
    print('Dùng CAR35 local đã kiểm tra; không tải/audit lại raw ZIP.')
else:
    if LOCAL_SOURCE_ZIP.strip():
        archive = Path(LOCAL_SOURCE_ZIP).expanduser()
        assert archive.is_file(), 'Không thấy LOCAL_SOURCE_ZIP; kiểm tra file còn trong runtime không.'
    else:
        assert SOURCE_URL.strip(), 'Điền SOURCE_URL bằng link ZIP 107 lớp.'
        archive = download_drive_zip(SOURCE_URL, WORK / 'downloads' / ('raw107_' + drive_file_id(SOURCE_URL) + '.zip'),
            expected_sha256=RAW_ZIP_SHA256)
    print('Kiểm tra ZIP và chuẩn bị dataset mới; không dùng hash/patch V4...', flush=True)
    dataset_info = prepare_raw_car35_zip(archive, DATASET_ROOT, dataset_version=DATASET_VERSION,
        seed=SEED, near_threshold=NEAR_HASH_DISTANCE, expected_source_sha256=RAW_ZIP_SHA256,
        workers=WORKERS, progress=progress)
manifest = json.loads((DATASET_ROOT / 'manifest.json').read_text(encoding='utf-8'))
receipt = dict(source_mode='raw107', archive_sha256=manifest['raw_zip_preparation']['source_zip_sha256'],
    dataset_version=DATASET_VERSION, manifest_sha256=dataset_info['manifest_sha256'])
(WORK / 'source_archive_verification.json').write_text(json.dumps(receipt, indent=2), encoding='utf-8')
print(f"CAR35: {dataset_info['images']:,} images | {dataset_info['boxes']:,} boxes | 35 classes")
for split in ('train','val','test'):
    print(split, ':', dataset_info['split_images'][split], 'images |', dataset_info['split_groups'][split], 'groups')
for split, names in manifest['missing_classes'].items():
    if names: print('Lớp thiếu ở', split, ':', ', '.join(names))
print('Manifest:', dataset_info['manifest_sha256'])
print('Test thực địa độc lập:', dataset_info['independent_field_test'])


## 3. EDA và kiểm tra nhãn trước khi train

Thống kê đúng CAR35 mới chuẩn bị, không coi đủ 35 tên lớp là đủ chất lượng từng lớp.
Rà phân bố train/val/test và nhãn mẫu; giữ manifest cùng kết quả để đối chiếu benchmark.


In [ ]:
import csv
import matplotlib.pyplot as plt
EDA_ROOT = WORK / 'eda'; EDA_ROOT.mkdir(exist_ok=True)
rows = [dict(class_id=i, name=name, **{s:dataset_info['class_counts'][s].get(i, 0) for s in ('train','val','test')})
        for i,name in enumerate(dataset_info['names'])]
with (EDA_ROOT / 'class_counts.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
fig, ax = plt.subplots(figsize=(12, 12)); left = [0]*35
for split in ('train','val','test'):
    values = [r[split] for r in rows]
    ax.barh(range(35), values, left=left, label=split)
    left = [a+b for a,b in zip(left,values)]
ax.set_yticks(range(35), dataset_info['names'], fontsize=8); ax.legend(); ax.set_title(dataset_info['dataset_version'] + ' boxes by split')
fig.tight_layout(); fig.savefig(EDA_ROOT / 'class_distribution.png', dpi=150); plt.close(fig)
(EDA_ROOT / 'summary.json').write_text(json.dumps(dataset_info, ensure_ascii=False, indent=2), encoding='utf-8')
for name in ('manifest.json','data.yaml','report.html','dataset_report.json','README.md'):
    if (DATASET_ROOT / name).is_file(): shutil.copy2(DATASET_ROOT / name, EDA_ROOT / name)
if (WORK / 'source_archive_verification.json').is_file():
    shutil.copy2(WORK / 'source_archive_verification.json', EDA_ROOT / 'source_archive_verification.json')
if (DATASET_ROOT / 'previews').is_dir(): shutil.copytree(DATASET_ROOT / 'previews', EDA_ROOT / 'previews', dirs_exist_ok=True)
for name in ('v4_manifest.json','label_patch.json'):
    source = DATASET_ROOT / 'provenance' / name
    if source.is_file():
        (EDA_ROOT / 'provenance').mkdir(exist_ok=True)
        shutil.copy2(source, EDA_ROOT / 'provenance' / name)
print('EDA saved:', EDA_ROOT)


In [ ]:
import random
from PIL import Image, ImageDraw
sample = manifest['images'].copy(); random.Random(SEED).shuffle(sample)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for axis in axes.flat: axis.axis('off')
for axis, row in zip(axes.flat, sample[:6]):
    with Image.open(DATASET_ROOT / row['image']) as source: image = source.convert('RGB')
    draw = ImageDraw.Draw(image)
    for line in (DATASET_ROOT / row['label']).read_text().splitlines():
        if not line.strip(): continue
        cls, x, y, w, h = map(float, line.split())
        box = ((x-w/2)*image.width, (y-h/2)*image.height, (x+w/2)*image.width, (y+h/2)*image.height)
        draw.rectangle(box, outline='yellow', width=2)
        draw.text((max(0,box[0]),max(0,box[1]-12)), CAR35[int(cls)], fill='yellow')
    axis.imshow(image); axis.set_title(row['split'])
fig.tight_layout(); fig.savefig(EDA_ROOT / 'bbox_preview.png', dpi=150); plt.show(); plt.close(fig)


## 4. Train YOLO11s

Bắt đầu từ pretrained yolo11s.pt; mặc định 640px, batch 8, 30 epoch và early stopping.
Nếu hết VRAM, giảm batch ở run mới. Resume yêu cầu cùng dataset/settings và last.pt còn trong runtime.
Best/last và checkpoint mỗi 5 epoch vẫn là file local; tải ZIP kết quả trước khi runtime bị xóa.


In [ ]:
if DEVICE != 'cpu':
    import torch
    assert torch.cuda.is_available(), 'Chọn Runtime → Change runtime type → GPU trước khi train.'
try:
    best_checkpoint = train_car35(DATASET_ROOT, RUN_DIR, epochs=EPOCHS, imgsz=IMAGE_SIZE,
        batch=BATCH, device=DEVICE, resume=RESUME)
    print('Best:', best_checkpoint)
except KeyboardInterrupt:
    print('Đã dừng. Chạy cell Xuất kết quả để giữ checkpoint nếu run đã tạo weights.')


## 3. Validation và test riêng

Validation được dùng để xem lỗi/chọn cấu hình. Test mặc định tắt; chỉ bật khi đã chốt.
Đủ 35 tên lớp không có nghĩa đủ mẫu kiểm thử. Giữ báo cáo giới hạn scene overlap và lớp hiếm khi đọc metric.


In [ ]:
from datetime import datetime, timezone
def fresh_evaluation_directory(label):
    output = RUN_DIR / label
    if output.exists():
        output = RUN_DIR / (label + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    return output
checkpoint = RUN_DIR / 'weights' / 'best.pt'
if checkpoint.is_file():
    val_report = evaluate_car35(checkpoint, DATASET_ROOT, fresh_evaluation_directory('validation'),
        imgsz=IMAGE_SIZE, batch=BATCH, device=DEVICE)
    print('Validation:', val_report['metrics'])
    if RUN_HELDOUT_TEST:
        test_report = evaluate_car35(checkpoint, DATASET_ROOT, fresh_evaluation_directory('test'), split='test', allow_test=True,
            imgsz=IMAGE_SIZE, batch=BATCH, device=DEVICE)
        print('Development test:', test_report['metrics'])
else:
    print('Chưa có best.pt để evaluate; có thể xuất last.pt nếu đã train một phần.')


## 4. Xuất kết quả về máy

ZIP gồm best.pt/last.pt, checkpoint định kỳ, plots/results.csv, training_request, evaluation reports và manifest/checksum.
Tải ZIP trước khi đóng runtime. Sau khi bạn gửi kết quả, mới rà chất lượng và mapping để tích hợp; notebook không đổi model đang phục vụ.


In [ ]:
from datetime import datetime, timezone
name = RUN_NAME + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
if RUN_DIR.is_dir() and (WORK / 'source_archive_verification.json').is_file():
    shutil.copy2(WORK / 'source_archive_verification.json', RUN_DIR / 'source_archive_verification.json')
if RUN_DIR.is_dir() and (WORK / 'eda').is_dir():
    shutil.copytree(WORK / 'eda', RUN_DIR / 'eda', dirs_exist_ok=True)
RESULT_ZIP = export_car35_run(RUN_DIR, DATASET_ROOT, WORK / (name + '.zip'))
print('Training result:', RESULT_ZIP)
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(RESULT_ZIP))
